In [ ]:
# ============================================================
# CELL 1 — INSTALL DEPENDENCIES
# ============================================================

%pip install -q \
    opencv-python \
    numpy \
    matplotlib \
    pillow \
    scipy \
    scikit-image \
    open3d \
    shapely \
    tqdm \
    requests \
    torch \
    torchvision \
    plotly

print("Base dependencies installed successfully.")


In [ ]:
# ============================================================
# CELL 2 — IMPORTS, CONFIGURATION & TELEMETRY BOOTSTRAP
# ============================================================
# WHAT: shared imports, the project directory layout, and the CARGO telemetry
#   layer (per-step logs, per-cell timing, live progress bars) that every later
#   cell uses. Nothing here changes pipeline behaviour.

import os
import cv2
import json
import math
import time as _cargo_time
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm

print("Libraries imported successfully.")

# ------------------------------------------------------------
# Project directories
# ------------------------------------------------------------

PROJECT_DIR = Path.cwd()

IMAGE_DIR = PROJECT_DIR / "images"
OUTPUT_DIR = PROJECT_DIR / "outputs"
MODEL_DIR = PROJECT_DIR / "models"

IMAGE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)

# ============================================================
# CARGO TELEMETRY  (timing / progress / live bars)
# ============================================================
# HOW TO USE
#   with cargo_timer("SAM encode", image=3):      ...     # timed block
#   cargo_note("something happened")                      # one-off log line
#   for img in cargo_stage("depth map", total=3):  ...     # live progress bar
#   cargo_timing_report()                                 # print the tables
#
# Switches (both read once, here):
#   CARGO_DEBUG=0    silence every log line (timings still recorded)
#   CARGO_PROGRESS=0 hide the live progress bars
# Both default to ON. Everything degrades to a no-op outside IPython
# (_build/_e2e_test.py execs these cells in a bare namespace), so the headless
# test harness keeps working untouched.

CARGO_DEBUG = os.environ.get("CARGO_DEBUG", "1").strip().lower() not in (
    "0", "false", "no", "off",
)
CARGO_PROGRESS = os.environ.get("CARGO_PROGRESS", "1").strip().lower() not in (
    "0", "false", "no", "off",
)
#: Rewrite outputs/timings.* after EVERY cell, not only at cell 10 / 28.
CARGO_LIVE_SAVE = os.environ.get("CARGO_LIVE_SAVE", "1").strip().lower() not in (
    "0", "false", "no", "off",
)

#: Total number of code cells, used by the whole-run progress bar.
CARGO_TOTAL_CELLS = 31

#: Every finished cargo_timer block lands here (dicts with name/seconds/...).
CARGO_STEP_LOG = []

#: Per-cell records, filled in by the IPython pre/post_run_cell hook.
CARGO_CELL_TIMINGS = []

#: Monotonic start of the whole run, used for the "elapsed" column.
CARGO_RUN_START = _cargo_time.perf_counter()

#: Stable id for this kernel session. Every live save reuses it, so the history
#: CSV can replace this run's rows instead of re-appending them each time.
CARGO_RUN_ID = _cargo_time.strftime("%Y%m%dT%H%M%SZ", _cargo_time.gmtime())

#: The previous run's timings.json, snapshotted before this run overwrites it.
CARGO_PREVIOUS_RUN = None

#: True once this session has looked for a previous run. The snapshot must be
#: taken exactly once -- on the first live save -- or a later save of THIS run
#: would be mistaken for the baseline.
CARGO_PREVIOUS_CAPTURED = False

#: The last pipeline cell: a run that reaches it has a complete profile.
CARGO_LAST_CELL = 28


def _cargo_stamp():
    """Wall-clock HH:MM:SS prefix for a log line."""
    return _cargo_time.strftime("%H:%M:%S")


def cargo_note(message):
    """Print one timestamped debug line (suppressed when CARGO_DEBUG=0)."""
    if CARGO_DEBUG:
        print(f"  [{_cargo_stamp()}] {message}", flush=True)


def cargo_step(message):
    """Announce the start of a step; returns the perf_counter start value."""
    if CARGO_DEBUG:
        print(f"  [{_cargo_stamp()}] >> {message}", flush=True)
    return _cargo_time.perf_counter()


def cargo_done(message, started, status="ok", **fields):
    """
    Close a step opened by cargo_step()/cargo_timer() and record it.

    `fields` are extra key=value pairs stored alongside the timing, e.g.
    cargo_done("detect", t0, objects=3, source="zip") -> 1.204s.
    """
    seconds = _cargo_time.perf_counter() - started
    record = {"name": message, "seconds": seconds, "status": status}
    record.update(fields)
    CARGO_STEP_LOG.append(record)

    if CARGO_DEBUG:
        details = "  ".join(f"{key}={value}" for key, value in fields.items())
        mark = "[ok]" if status == "ok" else "[!!]"
        suffix = f"  [{details}]" if details else ""
        print(f"  [{_cargo_stamp()}] {mark} {message} "
              f"({seconds:.3f}s){suffix}", flush=True)
    return seconds


class cargo_timer:
    """
    Context manager that times a block and logs it:

        with cargo_timer("SAM predict", roi="FULL") as t:
            ...
            t.note = {"masks": 3}

    `t.note` is merged into the recorded/logged fields.
    """

    def __init__(self, name, **fields):
        self.name = name
        self.fields = dict(fields)
        self.note = {}

    def __enter__(self):
        self.started = cargo_step(self.name)
        return self

    def __exit__(self, exc_type, exc, tb):
        fields = dict(self.fields)
        if self.note:
            fields.update(self.note)
        status = "ok" if exc_type is None else f"error:{exc_type.__name__}"
        cargo_done(self.name, self.started, status=status, **fields)
        return False


class cargo_stage:
    """
    Thin tqdm wrapper for a known-length loop -- the "live progress bar".

        for img in cargo_stage("depth map", total=len(images)):
            ...
            stage.set_note("image 2")

    Falls back to a plain iterator when CARGO_PROGRESS=0 or tqdm is unavailable,
    so the loop body never has to care whether a bar was drawn.
    """

    def __init__(self, description, total=None, unit="it", leave=True):
        self.description = description
        self.total = total
        self.unit = unit
        self.leave = leave
        self._bar = None
        self.n = 0

    def __enter__(self):
        self.started = cargo_step(self.description)
        if CARGO_PROGRESS:
            try:
                self._bar = tqdm(total=self.total, desc=self.description,
                                 unit=self.unit, leave=self.leave,
                                 dynamic_ncols=True)
            except Exception:          # never let a bar break the pipeline
                self._bar = None
        if CARGO_DEBUG and self.total is not None:
            print(f"        ({self.total} {self.unit} to go)", flush=True)
        return self

    def __iter__(self):
        source = range(self.total) if self.total is not None else iter(())
        for item in source:
            yield item
            self.update()

    def update(self, amount=1):
        self.n += amount
        if self._bar is not None:
            self._bar.update(amount)

    def set_note(self, text):
        """Show a short status on the bar (ignored when there is no bar)."""
        if self._bar is not None:
            self._bar.set_postfix_str(str(text)[:40], refresh=True)

    def __exit__(self, exc_type, exc, tb):
        if self._bar is not None:
            self._bar.close()
        status = "ok" if exc_type is None else f"error:{exc_type.__name__}"
        cargo_done(self.description, self.started, status=status,
                   items=self.n, unit=self.unit)

# ------------------------------------------------------------
# Live whole-run progress bar + automatic per-cell timing
# ------------------------------------------------------------
# Registers IPython's pre_run_cell / post_run_cell events, so EVERY cell in the
# notebook is timed without editing it. The raw cell source carries the
# "# CELL nn - TITLE" banner, which is what gets logged.
#
# No IPython shell (headless harness) -> registration is skipped and the rest of
# the notebook still runs; only the automatic per-cell numbers are missing.

CARGO_BAR = None          # the live ipywidgets.HTML bar, or None
CARGO_RUN_BAR_CELL = 0    # cells finished so far


def _cargo_cell_title(raw_cell):
    """Pull the '# CELL nn - ...' banner out of a cell's source."""
    for line in (raw_cell or "").splitlines():
        stripped = line.strip().lstrip("#").strip()
        if stripped.upper().startswith("CELL "):
            return stripped[:70]
    return "unnamed cell"


def _cargo_render_bar(done, total, title, seconds):
    """(Re)draw the whole-run progress bar in place."""
    if CARGO_BAR is None:
        return
    try:
        from IPython.display import display
        fraction = (done / total) if total else 0.0
        minutes, secs = divmod(int(seconds), 60)
        CARGO_BAR.value = (
            f"<b>{title}</b><br>"
            f"{'█' * int(fraction * 24):<24} {done}/{total} cells "
            f"· elapsed {minutes}m {secs:02d}s"
        )
        display(CARGO_BAR, display_id=True)
    except Exception:
        CARGO_BAR = None


def _cargo_pre_run_cell(info):
    """pre_run_cell hook: stamp the start time and announce the cell."""
    info.cargo_started = _cargo_time.perf_counter()
    if CARGO_DEBUG:
        title = _cargo_cell_title(getattr(info, "raw_cell", ""))
        stamp = _cargo_stamp()
        print(f"\n[{stamp}] {'=' * 62}\n"
              f"[{stamp}] >>> {title}\n"
              f"[{stamp}] {'=' * 62}", flush=True)


def _cargo_post_run_cell(info):
    """post_run_cell hook: record + log the cell duration, advance the bar."""
    started = getattr(info, "cargo_started", None)
    if started is None:
        return

    global CARGO_RUN_BAR_CELL
    seconds = _cargo_time.perf_counter() - started
    title = _cargo_cell_title(getattr(info, "raw_cell", ""))
    error = (getattr(info, "error_before_exec", None)
             or getattr(info, "error_in_exec", None))

    CARGO_CELL_TIMINGS.append({
        "cell": title,
        "seconds": seconds,
        "status": "error" if error is not None else "ok",
        "error": f"{type(error).__name__}: {error}" if error is not None else None,
    })
    CARGO_RUN_BAR_CELL += 1

    if CARGO_DEBUG:
        mark = "[!!] ERROR" if error is not None else "[ok]"
        print(f"[{_cargo_stamp()}] {mark} {title} -- {seconds:.3f}s", flush=True)

    _cargo_render_bar(CARGO_RUN_BAR_CELL, CARGO_TOTAL_CELLS, title,
                      _cargo_time.perf_counter() - CARGO_RUN_START)

    # Persist after EVERY cell, so the artefacts on disk are always current and
    # a run stopped at CELL 10 still leaves a record. Silent on purpose -- the
    # line above is the user-facing signal. Without an IPython shell (the test
    # harness) this hook never fires and the explicit CELL 10 / 28 calls take
    # over, so no path loses its timings.
    if CARGO_LIVE_SAVE:
        try:
            cargo_save_timings(announce=False)
        except Exception:
            pass


def _cargo_install_hooks():
    """Wire the hooks + the live bar. Returns False without an IPython shell."""
    global CARGO_BAR

    try:
        from IPython import get_ipython
        shell = get_ipython()
    except Exception:
        shell = None
    if shell is None:
        return False

    if CARGO_PROGRESS:
        try:
            import ipywidgets
            from IPython.display import display
            CARGO_BAR = ipywidgets.HTML(value="<b>Cargo pipeline</b><br>starting...")
            display(CARGO_BAR)
        except Exception:
            CARGO_BAR = None

    try:
        # Re-running CELL 2 must not stack a second pair of hooks, which would
        # double every timing and every live save.
        for event, handler in (("pre_run_cell", _cargo_pre_run_cell),
                               ("post_run_cell", _cargo_post_run_cell)):
            try:
                shell.events.unregister(event, handler)
            except Exception:
                pass
            shell.events.register(event, handler)
    except Exception:
        return False
    return True


CARGO_HOOKS_ACTIVE = _cargo_install_hooks()


# ------------------------------------------------------------
# Durable timing artefacts: outputs/timings.json + .md + .csv
# ------------------------------------------------------------
# The on-screen tables live in the kernel and die with the session, so
# cargo_save_timings() persists them next to the other run artifacts:
#
#   outputs/timings.json       full structured record of the latest run
#   outputs/timing_report.md   human-readable tables (per cell / per phase)
#   outputs/timing_history.csv one row per cell per run (re-written, not
#                              duplicated, while the run is still going)
#
# It runs automatically after EVERY cell via the post_run_cell hook above, so
# the files are always current -- nothing has to reach CELL 10 or CELL 28
# first, and an interrupted run still leaves a usable record. Set
# CARGO_LIVE_SAVE=0 to fall back to saving only at CELL 10 / CELL 28.
#
# A partial run (stopped at CELL 10) and a full run (CELL 28) are labelled
# differently and never compared against each other.

#: Bumped when the JSON layout changes; readers can branch on it.
CARGO_TIMING_SCHEMA = 1

#: (label, first cell, last cell) used for the per-phase roll-up.
#: CELL 10 is the LAST cell of the detect/segment stage; CELL 11+ is the
#: measurement stage (calibration, depth, 3D, volume, weight).
CARGO_PHASES = (
    ("Setup", 1, 6),
    ("Detect + Segment", 7, 10),
    ("Measurement", 11, 999),
)


def _cargo_output_dir():
    """Directory for the timing artefacts; tolerates being called early."""
    try:
        return Path(globals().get("OUTPUT_DIR") or (PROJECT_DIR / "outputs"))
    except Exception:
        return Path.cwd() / "outputs"


def _cargo_cell_number(banner):
    """'CELL 9b - TITLE' -> 9. Returns None when the banner is not a CELL one."""
    import re
    match = re.search(r"CELL\s+(\d+)", banner or "")
    return int(match.group(1)) if match else None


def _cargo_phase_of(banner):
    number = _cargo_cell_number(banner)
    if number is None:
        return "Other"
    for label, low, high in CARGO_PHASES:
        if low <= number <= high:
            return label
    return "Other"


def cargo_timing_payload(label="full run"):
    """Build the complete structured record for the current run."""
    import time as _t
    elapsed = _t.perf_counter() - CARGO_RUN_START
    return {
        "schema_version": CARGO_TIMING_SCHEMA,
        "run_id": CARGO_RUN_ID,
        "run_label": label,
        "kind": "full" if _cargo_reached_last_cell() else "live",
        "generated_utc": _t.strftime("%Y-%m-%dT%H:%M:%SZ", _t.gmtime()),
        "hooks_active": bool(CARGO_HOOKS_ACTIVE),
        "total_seconds": round(elapsed, 4),
        "cells_timed": len(CARGO_CELL_TIMINGS),
        "cells": [
            {
                "index": index,
                "cell": record["cell"],
                "cell_number": _cargo_cell_number(record["cell"]),
                "phase": _cargo_phase_of(record["cell"]),
                "seconds": round(float(record["seconds"]), 4),
                "status": record["status"],
                "error": record.get("error"),
            }
            for index, record in enumerate(CARGO_CELL_TIMINGS, start=1)
        ],
        "steps": [
            {
                "name": record.get("name", "?"),
                "seconds": round(float(record.get("seconds", 0.0)), 4),
                "status": record.get("status", "ok"),
                "details": {
                    key: value for key, value in record.items()
                    if key not in {"name", "seconds", "status"}
                },
            }
            for record in CARGO_STEP_LOG
        ],
        # Cell 9/10's own per-object dashboard, kept alongside the new timings.
        "roi_timings": [
            {
                roi: {key: value for key, value in data.items() if key != "decode_times"}
                for roi, data in per_image.items()
            }
            for per_image in (globals().get("roi_timings_by_image") or [])
        ],
    }


def _cargo_render_markdown(payload):
    """Build the human-readable timing_report.md body."""
    lines = [
        "# Cargo pipeline — execution timing",
        "",
        f"- run id: `{payload['run_id']}`  (label: **{payload['run_label']}**)",
        f"- generated (UTC): {payload['generated_utc']}",
        f"- per-cell hook: {'active' if payload['hooks_active'] else 'inactive (no IPython shell)'}",
        f"- cells timed: {payload['cells_timed']} of {CARGO_TOTAL_CELLS}",
        f"- total: {payload['total_seconds']:.2f} s",
        "",
    ]

    if not payload["cells"]:
        lines += [
            "> No per-cell timings were recorded in this session.",
            "> Run inside Jupyter/VS Code so the CELL 2 hook can time each cell.",
            "",
        ]
    else:
        lines += ["## Time per cell", "",
                  "| # | CELL | PHASE | TIME | STATUS |",
                  "|---|------|-------|------|--------|"]
        for cell in payload["cells"]:
            lines.append(
                f"| {cell['index']} | {cell['cell'][:44]} | {cell['phase']} "
                f"| {cell['seconds']:.3f}s | {cell['status']} |")
        lines.append("")

        # --- per-phase roll-up ----------------------------------------------
        lines += ["## Time per phase", "", "| PHASE | CELLS | TOTAL | SHARE |",
                  "|---|---|---|---|"]
        for label, _low, _high in CARGO_PHASES:
            members = [c for c in payload["cells"] if c["phase"] == label]
            if not members:
                continue
            total = sum(c["seconds"] for c in members)
            share = (total / payload["total_seconds"] * 100.0) if payload["total_seconds"] else 0.0
            lines.append(f"| {label} | {len(members)} | {total:.3f}s | {share:.1f}% |")
        lines.append("")

    if payload["steps"]:
        lines += ["## Slowest steps (top 20)", "",
                  "| STEP | TIME | STATUS |", "|--------|------|--------|"]
        for step in sorted(payload["steps"], key=lambda s: -s["seconds"])[:20]:
            lines.append(
                f"| {step['name'][:44]} | {step['seconds']:.3f}s | {step['status']} |")
        lines.append("")

    if payload["roi_timings"]:
        lines += ["## Per-object ROI timings (CELL 9)", "",
                  "| IMAGE | ROI | OBJECTS | SOURCE | TOTAL |",
                  "|-------|-----|---------|--------|-------|"]
        for image_index, per_image in enumerate(payload["roi_timings"], start=1):
            for roi, data in per_image.items():
                lines.append(
                    f"| {image_index} | {roi} | {data.get('objects', 0)} "
                    f"| {str(data.get('source', '-'))[:24]} "
                    f"| {data.get('total_seconds', 0.0):.3f}s |")
        lines.append("")

    return "\n".join(lines) + "\n"


def _cargo_write_history(payload, path):
    """
    Merge this run's per-cell rows into the history CSV.

    Rows belonging to the CURRENT run are replaced (every live save re-emits the
    cells it already has) while every other run's rows are preserved, so the
    file grows once per run instead of once per save.
    """
    import csv

    fieldnames = ["run_id", "run_label", "generated_utc",
                  "cell_index", "cell", "phase", "seconds", "status"]

    kept = []
    if path.exists():
        with open(path, "r", newline="", encoding="utf-8") as handle:
            for row in csv.reader(handle):
                if not row or row[0] == "run_id":
                    continue
                if row[0] != payload["run_id"]:
                    kept.append(row)

    for cell in payload["cells"]:
        kept.append([
            payload["run_id"], payload["run_label"], payload["generated_utc"],
            str(cell["index"]), cell["cell"], cell["phase"],
            f"{cell['seconds']:.4f}", cell["status"],
        ])

    with open(path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(fieldnames)
        writer.writerows(kept)


def _cargo_reached_last_cell():
    """True once the final pipeline cell (CELL 28) has been timed."""
    for record in CARGO_CELL_TIMINGS:
        number = _cargo_cell_number(record["cell"])
        if number is not None and number >= CARGO_LAST_CELL:
            return True
    return False


def cargo_auto_label(timed=None):
    """
    Honest label for a save made part-way through a run.

    Counting cells is not enough: CELL 2 installs the hook, so the notebook's
    first cell is never timed and the count can never reach CARGO_TOTAL_CELLS.
    Completion is therefore "did the run reach CELL 28?".
    """
    if _cargo_reached_last_cell():
        return "full run"
    return "live (%d cells)" % (len(CARGO_CELL_TIMINGS) if timed is None else timed)


def _cargo_run_kind(payload):
    """Coarse run class -- 'full' or 'live' -- used to gate comparisons."""
    if not payload:
        return None
    kind = payload.get("kind")
    if kind:
        return kind
    return "full" if payload.get("run_label") == "full run" else "live"


def _cargo_capture_previous():
    """
    Snapshot the PREVIOUS run's timings.json before this run overwrites it.

    Live saving rewrites timings.json after every cell, so the first save of the
    session is the last chance to keep the baseline. A file written by this same
    kernel session is not a baseline -- it is this run.
    """
    global CARGO_PREVIOUS_RUN, CARGO_PREVIOUS_CAPTURED
    if CARGO_PREVIOUS_CAPTURED:
        return
    CARGO_PREVIOUS_CAPTURED = True
    try:
        path = _cargo_output_dir() / "timings.json"
        if not path.exists():
            return
        record = json.loads(path.read_text(encoding="utf-8"))
        if record.get("run_id") == CARGO_RUN_ID:
            return
        CARGO_PREVIOUS_RUN = record
    except Exception:
        CARGO_PREVIOUS_RUN = None


def cargo_save_timings(label=None, announce=True):
    """
    Persist the run's timings to OUTPUT_DIR.

    Driven automatically by the CELL 2 post_run_cell hook, so the files on disk
    are current after EVERY cell -- nothing has to reach CELL 10 or CELL 28
    first, and a run that is interrupted still leaves a record behind.

    ``label=None`` auto-labels the save: "full run" once every cell has been
    timed, otherwise "live (n cells)". Never raises -- a telemetry write must
    not be able to break a working pipeline.
    """
    try:
        _cargo_capture_previous()
        if label is None:
            label = cargo_auto_label()
        payload = cargo_timing_payload(label)
        out_dir = _cargo_output_dir()
        out_dir.mkdir(parents=True, exist_ok=True)

        json_path = out_dir / "timings.json"
        md_path = out_dir / "timing_report.md"
        csv_path = out_dir / "timing_history.csv"

        json_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
        md_path.write_text(_cargo_render_markdown(payload), encoding="utf-8")
        _cargo_write_history(payload, csv_path)

        if announce:
            print(f"  [{_cargo_stamp()}] [ok] timings saved [{label}] -> "
                  f"{json_path.name}, {md_path.name}, {csv_path.name} "
                  f"({payload['cells_timed']} cells, "
                  f"{payload['total_seconds']:.2f}s)", flush=True)
        return {
            "json": str(json_path), "markdown": str(md_path),
            "history": str(csv_path), "payload": payload,
        }
    except Exception as exc:                      # never break the pipeline
        print(f"  [!!] timing save failed: {type(exc).__name__}: {exc}", flush=True)
        return None


def cargo_compare_timings(current=None, previous=None):
    """
    Print a per-cell delta between this run and the previous one.

    Only runs of the same KIND are compared, so a partial run (stopped at
    CELL 10) is never diffed against a full one. Pass either payload
    explicitly, or omit them to use the baseline snapshotted at the start of
    this run and, failing that, the timings.json on disk.
    """
    if current is None:
        current = cargo_timing_payload()
    if previous is None:
        # Prefer the baseline snapshotted before this run's first live save: the
        # file on disk has already been overwritten by this run's own saves.
        previous = CARGO_PREVIOUS_RUN
    if previous is None:
        try:
            previous = json.loads(
                (_cargo_output_dir() / "timings.json").read_text(encoding="utf-8"))
        except Exception:
            previous = None

    # The guard applies to an explicitly passed baseline too: a run that never
    # reached the last cell must never be diffed against a complete one.
    if previous and _cargo_run_kind(previous) != _cargo_run_kind(current):
        cargo_note(f"skipping comparison: previous run is "
                   f"'{previous.get('run_label')}', current is '{current['run_label']}'")
        return False

    if not previous:
        cargo_note("no previous run with the same label to compare against")
        return False

    before = {c["cell"]: c["seconds"] for c in previous.get("cells", [])}
    rows = []
    for cell in current["cells"]:
        if cell["cell"] not in before:
            continue
        old, new = before[cell["cell"]], cell["seconds"]
        delta = new - old
        pct = (delta / old * 100.0) if old > 1e-9 else (100.0 if new > 0 else 0.0)
        rows.append((cell["cell"], old, new, delta, pct))

    print("\n" + "=" * 74)
    print(f"TIMING COMPARISON  {previous['run_id']}  ->  {current['run_id']}  "
          f"({current['run_label']})")
    print("=" * 74)
    print(f"{'CELL':<44} {'PREV':>9} {'NOW':>9} {'DELTA':>9} {'%':>8}")
    print("-" * 74)
    for name, old, new, delta, pct in rows:
        print(f"{name[:44]:<44} {old:>8.3f}s {new:>8.3f}s {delta:>+8.3f}s {pct:>+7.1f}%")
    print("-" * 74)
    old_total = sum(before.values())
    new_total = sum(c["seconds"] for c in current["cells"])
    print(f"{'TOTAL (cells only)':<44} {old_total:>8.3f}s {new_total:>8.3f}s "
          f"{new_total - old_total:>+8.3f}s")
    print("=" * 74)
    return True


def cargo_timing_report(partial=False):
    """Print the per-cell and per-step timing tables collected so far."""
    elapsed = _cargo_time.perf_counter() - CARGO_RUN_START
    minutes, seconds = divmod(int(elapsed), 60)

    print("\n" + "=" * 74)
    if partial:
        print("EXECUTION TIME PER CELL  (partial — run to CELL 28 for the full profile)")
    else:
        print("EXECUTION TIME PER CELL")
    print("=" * 74)
    print(f"{'#':>3}  {'CELL':<46} {'TIME':>10}  STATUS")
    print("-" * 74)
    for index, record in enumerate(CARGO_CELL_TIMINGS, start=1):
        print(f"{index:>3}  {record['cell'][:46]:<46} "
              f"{record['seconds']:>9.3f}s  {record['status']}")
    print("-" * 74)
    hook_note = "" if CARGO_HOOKS_ACTIVE else "   (per-cell hook inactive: no IPython shell)"
    print(f"total: {len(CARGO_CELL_TIMINGS)} cells in "
          f"{minutes}m {seconds:02d}s{hook_note}")
    print("=" * 74)

    if not CARGO_STEP_LOG:
        return

    print("\nSLOWEST STEPS (top 15)")
    print(f"{'STEP':<46} {'TIME':>10}")
    print("-" * 74)
    for record in sorted(CARGO_STEP_LOG, key=lambda r: -r["seconds"])[:15]:
        print(f"{record['name'][:46]:<46} {record['seconds']:>9.3f}s")
    print("=" * 74)


if CARGO_DEBUG:
    print(f"Telemetry: debug={CARGO_DEBUG} progress={CARGO_PROGRESS} "
          f"(set CARGO_DEBUG=0 / CARGO_PROGRESS=0 to quiet)")
    print(f"Live bar + per-cell timing: "
          f"{'ACTIVE' if CARGO_HOOKS_ACTIVE else 'unavailable (no IPython shell)'}")



In [ ]:
# ============================================================
# CELL 3 — SELECT LOCAL CARGO IMAGES
# ============================================================

import os
import shutil
import tkinter as tk
from tkinter import filedialog

print("Select 1-3 cargo images from your computer.")
print("Ideally, select images from different viewing angles.")

image_source_dir = PROJECT_DIR / "Cargo Detection PoC"

# Headless/driver override: CARGO_IMAGES="path1.png;path2.jpg" runs without the
# Tk file dialog. When unset, the normal interactive picker opens as before.
_images_override = os.environ.get("CARGO_IMAGES", "").strip()

if _images_override:
    selected_files = tuple(
        p for p in (part.strip() for part in _images_override.split(";")) if p
    )
    print(f"[CARGO_IMAGES] using {len(selected_files)} image(s) — file dialog skipped.")
else:
    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)

    selected_files = filedialog.askopenfilenames(
        title="Select cargo images",
        initialdir=str(image_source_dir if image_source_dir.exists() else PROJECT_DIR),
        filetypes=[
            ("Image files", "*.jpg *.jpeg *.png *.jfif *.bmp *.webp"),
            ("All files", "*.*")
        ]
    )

    root.destroy()

if not selected_files:
    raise RuntimeError("No images were selected.")

image_paths = []

_t3 = cargo_step(f"3. stage {len(selected_files)} selected image(s)")

with cargo_stage("3. stage images",total=len(selected_files),unit="file",
                 leave=False) as copy_stage:
 for selected_file in selected_files:
    copy_stage.set_note(Path(selected_file).name[:36])
    source_path = Path(selected_file)
    output_path = IMAGE_DIR / source_path.name
    if source_path.resolve() != output_path.resolve():
        shutil.copy2(source_path, output_path)
    image_paths.append(str(output_path))
    copy_stage.update()

cargo_done("3. stage selected images", _t3, staged=len(image_paths))

print("\nSelected images:")

for path in image_paths:
    print(" -", path)

if len(image_paths) > 3:
    print("\nWARNING: PoC supports 1-3 images.")


In [ ]:
# ============================================================
# CELL 4 — VISUALIZE INPUT IMAGES
# ============================================================

def load_image(path):
    image = cv2.imread(path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    return image


images = []

with cargo_stage("4. load input images",total=len(image_paths),unit="image") as load_stage:
 for path in image_paths:
    load_stage.set_note(Path(path).name[:36])
    _t4 = cargo_step("  decode image")
    img = load_image(path)
    images.append(img)
    cargo_done("  decode image", _t4, shape=f"{img.shape[1]}x{img.shape[0]}")
    load_stage.update()

fig, axes = plt.subplots(
    1,
    len(images),
    figsize=(7 * len(images), 6)
)

if len(images) == 1:
    axes = [axes]

for ax, img, path in zip(axes, images, image_paths):

    ax.imshow(img)
    ax.set_title(Path(path).name)
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# CELL 5 — IMAGE QUALITY CHECK + PREPROCESSING
# ============================================================

def image_quality_report(image):

    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

    # Blur score using Laplacian variance
    blur_score = cv2.Laplacian(gray, cv2.CV_64F).var()

    # Brightness
    brightness = np.mean(gray)

    # Resolution
    height, width = image.shape[:2]

    return {
        "width": width,
        "height": height,
        "blur_score": float(blur_score),
        "brightness": float(brightness)
    }


with cargo_stage("5. image quality check",total=len(images),unit="image") as q_stage:
 for idx, image in enumerate(images):
    q_stage.set_note(f"image {idx+1}/{len(images)}")
    _t5 = cargo_step(f"  quality metrics [image {idx+1}]")

    report = image_quality_report(image)
    cargo_done(f"  quality metrics [image {idx+1}]", _t5,
        blur=f"{report['blur_score']:.1f}",bright=f"{report['brightness']:.1f}")

    print(f"\nImage {idx + 1}")
    print("-" * 40)

    for key, value in report.items():
        print(f"{key}: {value:.2f}" if isinstance(value, float) else f"{key}: {value}")
    q_stage.update()


In [ ]:
# ============================================================
# CELL 6 — INSTALL SAM 3 (Segment Anything with Concepts) + SAM 1 (ViT-B, fallback)
# ============================================================
# WHAT: install SAM 3 (text/concept-promptable) as the PRIMARY segmenter, and keep
#   the original SAM 1 ViT-B as an automatic fallback so the notebook never dies.
# WHY THIS CELL IS DEFENSIVE: three failure modes are handled explicitly --
#   1. re-running this cell must not abort on an existing sam3/ clone,
#   2. Cell 1 installs unpinned numpy (2.x) but sam3 requires numpy>=1.26,<2,
#   3. the SAM 3 checkpoint comes from a gated HF repo and is cached locally so
#      it is fetched once, not on every run.
import os as _c6os, sys as _c6sys, subprocess as _c6sub

PROJECT_DIR = globals().get("PROJECT_DIR", Path.cwd())
PROJECT_DIR = Path(PROJECT_DIR)
MODEL_DIR = Path(globals().get("MODEL_DIR", PROJECT_DIR / "models")); MODEL_DIR.mkdir(parents=True, exist_ok=True)

def _c6run(*args):
    """pip via the CURRENT interpreter -- never a bare `pip`, which may target another env."""
    r = _c6sub.run([_c6sys.executable, "-m", "pip", "install", "-q", *args],
                   capture_output=True, text=True)
    return r.returncode == 0, (r.stderr or r.stdout or "")[-600:]

# ---- 6a. numpy < 2 (SAM 3 hard requirement: sam3/pyproject.toml -> numpy>=1.26,<2) ----
import numpy as _c6np
_major = int(_c6np.__version__.split(".")[0])
if _major >= 2:
    print(f"numpy {_c6np.__version__} detected -> downgrading to <2 (SAM 3 requirement).")
    _c6run("numpy<2")
    import importlib as _c6il; _c6il.reload(_c6np)
    print("numpy now:", _c6np.__version__)

# ---- 6b. clone SAM 3 (guarded: re-running must not fail on an existing checkout) ----
SAM3_DIR = PROJECT_DIR / "sam3"
if (SAM3_DIR / "sam3" / "model_builder.py").exists():
    print("SAM 3 source already present -> skipping clone.")
else:
    if not _c6os.access(SAM3_DIR, _c6os.W_OK):
        print(f"[warn] '{SAM3_DIR}' not writable; cloning SAM 3 into a temp dir instead.")
        import tempfile as _c6tf
        SAM3_DIR = Path(_c6tf.mkdtemp(prefix="sam3_"))
    if not (SAM3_DIR / ".git").exists():
        r = _c6sub.run(["git", "clone", "--depth", "1",
                        "https://github.com/facebookresearch/sam3.git", str(SAM3_DIR)],
                       capture_output=True, text=True)
        if r.returncode != 0:
            print("[warn] git clone failed:", (r.stderr or "")[-300:])

# ---- 6c. editable install (+ HF hub for the checkpoint) ----
_c6ok, _c6err = _c6run("huggingface_hub")
if not _c6ok:
    print("[warn] huggingface_hub install failed:", _c6err)

if (SAM3_DIR / "pyproject.toml").exists():
    _c6ok, _c6err = _c6run("-e", str(SAM3_DIR))
    print("SAM 3 editable install:", "ok" if _c6ok else f"FAILED {_c6err}")

# ---- 6d. gated-repo token (Colab secrets first, then env) ----
SAM3_HF_TOKEN = _c6os.environ.get("HF_TOKEN") or _c6os.environ.get("HUGGINGFACE_HUB_TOKEN")
try:
    from google.colab import userdata as _c6ud          # Colab-only
    SAM3_HF_TOKEN = SAM3_HF_TOKEN or _c6ud.get("HF_TOKEN")
except Exception:
    pass
if SAM3_HF_TOKEN:
    _c6os.environ["HF_TOKEN"] = SAM3_HF_TOKEN
    print("HF_TOKEN found -> gated SAM 3 checkpoint is accessible.")
else:
    print("No HF_TOKEN set -> relying on public cache access for facebook/sam3.")

# ---- 6e. VERIFY, don't just print (a silent ImportError here would surface in Cell 9) ----
SAM3_INSTALL_OK = False
try:
    import sam3
    from sam3 import build_sam3_image_model
    from sam3.model.sam3_image_processor import Sam3Processor
    SAM3_INSTALL_OK = True
    print("SAM 3 import OK. version:", getattr(sam3, "__version__", "?"),
          "| source:", Path(sam3.__file__).resolve().parent)
except Exception as _c6e:
    print("[WARN] SAM 3 unavailable:", type(_c6e).__name__, _c6e)
    print("       Cells 7/9 will fall back to SAM 1 (ViT-B).")

# ---- 6f. SAM 1 (ViT-B) fallback -- unchanged install, still required for the fallback ----
if not _c6run("git+https://github.com/facebookresearch/segment-anything.git")[0]:
    _c6sub.run([_c6sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/facebookresearch/segment-anything.git"],
               capture_output=True, text=True)

#: "sam3" = concept-promptable (default) | "sam1" = force the legacy ViT-B path.
SAM_BACKEND = "sam3"
SAM3_DIR = SAM3_DIR
print(f"Cell 6 ready | SAM 3 available: {SAM3_INSTALL_OK} | backend: {SAM_BACKEND}")


In [ ]:
# ============================================================
# CELL 7 — LOAD MODELS: SAM ViT-B + CARGO-SPECIFIC YOLO (from the Cargo zip dataset)
# ============================================================
# WHAT: init custom YOLO11 (primary detector) + SAM ViT-B + OpenCV utils.
# VERIFY: weight files exist, class names loaded, image dims read dynamically.
# No hard-coded image W/H. No manual clicks. No external vision API.
import os, json, random, shutil, zipfile, requests
import numpy as np
import cv2
from pathlib import Path
from tqdm.auto import tqdm
import torch

PROJECT_DIR = globals().get("PROJECT_DIR", Path.cwd())
PROJECT_DIR = Path(PROJECT_DIR)
MODEL_DIR = globals().get("MODEL_DIR", PROJECT_DIR / "models")
MODEL_DIR = Path(MODEL_DIR); MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = globals().get("OUTPUT_DIR", PROJECT_DIR / "outputs")
OUTPUT_DIR = Path(OUTPUT_DIR); OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- 7a. Segmenter: SAM 3 (concept-promptable, PRIMARY) with SAM 1 ViT-B fallback ----
# SAM 3 accepts TEXT ("concept") prompts and +/- BOX prompts; SAM 1 only accepts
# pixel boxes/points. That extra expressiveness is the detection-rate win, so SAM 3
# is built first and the SAM 1 checkpoint is only downloaded if SAM 3 fails.
SAM_BACKEND = globals().get("SAM_BACKEND", "sam3")     # "sam3" | "sam1"
SAM3_AVAILABLE = bool(globals().get("SAM3_INSTALL_OK", False))
SAM3_CONF_THRESHOLD = 0.20     # Sam3Processor keeps instances above this. Kept low on
                               # PURPOSE: it is the only recall gate SAM 3 has.
                               # Precision is enforced downstream by Cell 10's
                               # MIN_ACCEPT_SCORE / border-leak gates.
SAM3_DIR = Path(globals().get("SAM3_DIR", PROJECT_DIR / "sam3"))
sam3_model = None; sam3_processor = None
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    torch.set_num_threads(4)

def _try_build_sam3():
    """Build SAM 3. Returns (model, processor, error_string). NEVER raises."""
    if not SAM3_AVAILABLE:
        return None, None, "sam3 package not importable"
    try:
        import sam3 as _s3
        from sam3 import build_sam3_image_model
        from sam3.model.sam3_image_processor import Sam3Processor
        bpe = Path(SAM3_DIR) / "assets" / "bpe_simple_vocab_16e6.txt.gz"
        if not bpe.exists():        # editable install ships it; else use the package copy
            bpe = Path(_s3.__file__).resolve().parent / "assets" / "bpe_simple_vocab_16e6.txt.gz"
        _t7s = cargo_step("7a. build SAM 3 (concept prompts)")
        # Cache the gated checkpoint locally so it is fetched ONCE, not every run.
        local_ckpt = MODEL_DIR / "sam3.pt"
        ckpt = str(local_ckpt) if local_ckpt.exists() else None
        model = build_sam3_image_model(bpe_path=str(bpe), device=device,
                                       checkpoint_path=ckpt, load_from_HF=(ckpt is None))
        try:
            if ckpt is None:
                from sam3.model_builder import download_ckpt_from_hf
                resolved = download_ckpt_from_hf(version="sam3")
                if resolved and Path(resolved).exists():
                    shutil.copy2(resolved, local_ckpt)
                    print("SAM 3 checkpoint cached ->", local_ckpt.name)
        except Exception:
            pass    # caching is an optimisation only; the model is already built
        proc = Sam3Processor(model, confidence_threshold=SAM3_CONF_THRESHOLD)
        cargo_done("7a. build SAM 3 (concept prompts)", _t7s,
                   device=device, conf=SAM3_CONF_THRESHOLD)
        return model, proc, None
    except Exception as e:
        return None, None, f"{type(e).__name__}: {e}"

if SAM_BACKEND == "sam3":
    sam3_model, sam3_processor, _err = _try_build_sam3()
    if sam3_processor is not None:
        print(f"SAM 3 ready on {device} (text + box prompts, conf>{SAM3_CONF_THRESHOLD}).")
    else:
        print(f"[WARN] SAM 3 build failed ({_err}) -> falling back to SAM 1 ViT-B.")
        SAM_BACKEND = "sam1"

# ---- SAM 1 ViT-B: used when SAM_BACKEND=="sam1" (explicit or automatic fallback) ----
sam = None; predictor = None
if SAM_BACKEND == "sam1":
    _t7 = cargo_step("7a. SAM ViT-B checkpoint")
    SAM_CHECKPOINT = MODEL_DIR / "sam_vit_b_01ec64.pth"
    SAM_URL = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"
    if not SAM_CHECKPOINT.exists():
        print("Downloading SAM ViT-B checkpoint...")
        r = requests.get(SAM_URL, stream=True); r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        with open(SAM_CHECKPOINT, "wb") as f:
            with tqdm(total=total, unit="B", unit_scale=True, desc="SAM ViT-B",
                      dynamic_ncols=True) as p:
                for ch in r.iter_content(chunk_size=1024 * 1024):
                    if ch: f.write(ch); p.update(len(ch))
        print("SAM checkpoint saved:", SAM_CHECKPOINT.stat().st_size, "bytes")
        cargo_done("7a. SAM checkpoint download", _t7, mb=f"{SAM_CHECKPOINT.stat().st_size/1e6:.1f}")
    else:
        print("SAM checkpoint already exists.")
        cargo_done("7a. SAM checkpoint (cached)", _t7)
    assert SAM_CHECKPOINT.exists(), "SAM checkpoint missing!"

    from segment_anything import sam_model_registry, SamPredictor
    _t7b = cargo_step("7a. build SAM ViT-B model")
    sam = sam_model_registry["vit_b"](checkpoint=str(SAM_CHECKPOINT))
    if device == "cpu":
        try: torch.backends.mkldnn.enabled = True
        except Exception: pass
        try: sam.to(memory_format=torch.channels_last)
        except Exception: pass
    sam.to(device); sam.eval()
    predictor = SamPredictor(sam)
    cargo_done("7a. build SAM ViT-B model", _t7b, device=device)
    print("SAM 1 (ViT-B) loaded on:", device)

assert predictor is not None or sam3_processor is not None, \
    "No segmenter available: both SAM 3 and SAM 1 failed to load."
USE_SAM3 = (SAM_BACKEND == "sam3" and sam3_processor is not None)
print("Segmenter:", "SAM 3 (concept prompts)" if USE_SAM3 else "SAM 1 (ViT-B, geometric prompts)")

# ---- 7b. Cargo taxonomy (Roboflow 8-class) ----
CARGO_CLASS_NAMES = ["Cargo-Box", "Cargo-Detection", "Chain", "Crane",
                     "Hook", "Net", "Strap", "Wooden-Plate"]
CARGO_CLASS_TARGETS = ("Cargo-Box", "Wooden-Plate")   # solid payload -> segment
CARGO_CLASS_ENVELOPE = ("Cargo-Detection",)           # net+load -> fallback only
CARGO_CLASS_EXCLUSIONS = ("Net", "Strap", "Chain", "Crane", "Hook")  # suppress
print("Class names:", CARGO_CLASS_NAMES)
# ---- 7c. dataset repair (flat train/ -> images|labels/{train,val}) ----
CARGO_ZIP_PATH = PROJECT_DIR / "Cargo Detection PoC" / "Cargo-Detection.v3i.yolov8-obb.zip"
CARGO_DATASET_DIR = PROJECT_DIR / "cargo_roboflow_obb_dataset"
CARGO_DATA_YAML = PROJECT_DIR / "cargo_obb_data.yaml"
CARGO_VAL_RATIO = 0.30; CARGO_SPLIT_SEED = 42
def _cargo_scene_key(p):
    """Source-scene key for a Roboflow export.

    Roboflow writes augmented variants as '<original>.rf.<hash>.jpg', so the stem
    before the first '.rf.' identifies the underlying photo. Splitting on SCENES
    (not files) keeps near-identical twins on one side of train/val -- otherwise
    validation mAP measures memorisation instead of generalisation."""
    return p.stem.split(".rf.")[0]
def prepare_cargo_zip_dataset(force=False):
    src_images = CARGO_DATASET_DIR / "train" / "images"
    src_labels = CARGO_DATASET_DIR / "train" / "labels"
    if force or not src_images.exists():
        if not CARGO_ZIP_PATH.exists():
            raise FileNotFoundError(f"Cargo zip not found: {CARGO_ZIP_PATH}")
        CARGO_DATASET_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(CARGO_ZIP_PATH) as z: z.extractall(CARGO_DATASET_DIR)
    it, iv, lt, lv = (CARGO_DATASET_DIR/"images"/"train", CARGO_DATASET_DIR/"images"/"val",
                      CARGO_DATASET_DIR/"labels"/"train", CARGO_DATASET_DIR/"labels"/"val")
    for d in (it, iv, lt, lv): d.mkdir(parents=True, exist_ok=True)
    if not any(it.glob("*")) and not any(iv.glob("*")):
        found = sorted([p for p in src_images.glob("*") if p.is_file()
                        and p.suffix.lower() in {".jpg",".jpeg",".png",".bmp",".webp"}])
        if not found: raise FileNotFoundError(f"No images in {src_images}")
        # Group augmented variants by source scene; split on SCENES, not files.
        _scenes = {}
        for p in found: _scenes.setdefault(_cargo_scene_key(p), []).append(p)
        _names = sorted(_scenes); random.Random(CARGO_SPLIT_SEED).shuffle(_names)
        _nval_scenes = max(1, int(round(len(_names)*CARGO_VAL_RATIO)))
        _val_scenes = set(_names[:_nval_scenes])
        vs = {p for k in _val_scenes for p in _scenes[k]}
        _t7c = cargo_step("7c. dataset split copy")
        with cargo_stage("    copy images",total=len(found),unit="file",leave=False) as cp:
            for p in found:
                cp.set_note(p.name[:36])
                ls = src_labels / f"{p.stem}.txt"; v = p in vs
                shutil.copy2(p, (iv if v else it)/p.name)
                if ls.exists(): shutil.copy2(ls, (lv if v else lt)/ls.name)
                cp.update()
        cargo_done("7c. dataset split copy", _t7c, files=len(found), val=len(vs))
    nt, nv = len(list(it.glob("*"))), len(list(iv.glob("*")))
    if nt == 0 or nv == 0: raise RuntimeError(f"Bad split train={nt} val={nv}")
    # Guard the on-disk split: a scene on both sides makes val mAP meaningless.
    _vt = {_cargo_scene_key(p) for p in iv.glob("*") if p.is_file()}
    _tr = {_cargo_scene_key(p) for p in it.glob("*") if p.is_file()}
    if _vt & _tr:
        raise RuntimeError(
            f"train/val scene leakage ({len(_vt & _tr)} scenes, e.g. "
            f"{sorted(_vt & _tr)[:2]}) - delete '{CARGO_DATASET_DIR}/images' and "
            f"re-run Cell 7 to rebuild a scene-grouped split")
    CARGO_DATA_YAML.write_text("\n".join(["# Auto-generated by Cell 7",
        f"path: {CARGO_DATASET_DIR.as_posix()}", "train: images/train",
        "val: images/val", "", "names:"] +
        [f"  {i}: {n}" for i, n in enumerate(CARGO_CLASS_NAMES)])+"\n", encoding="utf-8")
    return {"dataset_dir": CARGO_DATASET_DIR, "yaml": CARGO_DATA_YAML,
            "train": nt, "val": nv,
            "train_scenes": len(_tr), "val_scenes": len(_vt)}
try:
    _t7d = cargo_step("7c. prepare cargo zip dataset")
    cargo_dataset_info = prepare_cargo_zip_dataset()
    cargo_done("7c. prepare cargo zip dataset", _t7d,
        train=cargo_dataset_info["train"],val=cargo_dataset_info["val"])
    print(f"Cargo zip dataset ready -> {cargo_dataset_info['train']} train / "
          f"{cargo_dataset_info['val']} val images "
          f"({cargo_dataset_info['train_scenes']} / {cargo_dataset_info['val_scenes']} scenes, "
          f"scene-grouped) | yaml: {CARGO_DATA_YAML.name}")
except Exception as e:
    cargo_dataset_info = None
# ---- 7d. Custom YOLO11 weights (PRIMARY) with safe fallback ----
# Priority: cargo_yolo11_best.pt (custom YOLO11) > cargo_obb_best.pt (zip OBB
#   helper/exclusions) > yolo11m.pt > yolo11x.pt (generic COCO fallback).
CARGO_YOLO11_WEIGHTS = MODEL_DIR / "cargo_yolo11_best.pt"
_t7e = cargo_step("7d. resolve YOLO weights")
CARGO_OBB_WEIGHTS = MODEL_DIR / "cargo_obb_best.pt"
CARGO_FALLBACK_WEIGHTS = "yolo11m.pt"
AUTO_TRAIN_FROM_ZIP = False
if AUTO_TRAIN_FROM_ZIP and cargo_dataset_info is not None and not CARGO_YOLO11_WEIGHTS.exists():
    print("AUTO_TRAIN_FROM_ZIP=True -> run train_cargo_yolo11.py (standalone) instead.")
if CARGO_YOLO11_WEIGHTS.exists():
    CARGO_YOLO_WEIGHTS, CARGO_MODEL_ORIGIN = CARGO_YOLO11_WEIGHTS, "custom_yolo11"
elif CARGO_OBB_WEIGHTS.exists():
    CARGO_YOLO_WEIGHTS, CARGO_MODEL_ORIGIN = CARGO_OBB_WEIGHTS, "zip_trained_obb"
else:
    CARGO_YOLO_WEIGHTS, CARGO_MODEL_ORIGIN = CARGO_FALLBACK_WEIGHTS, "coco_pretrained_fallback"
print(f"Cargo YOLO detector: {CARGO_YOLO_WEIGHTS} (origin: {CARGO_MODEL_ORIGIN})")
cargo_done("7d. resolve YOLO weights", _t7e, origin=CARGO_MODEL_ORIGIN)
if CARGO_MODEL_ORIGIN != "custom_yolo11":
    print("  -> Train custom YOLO11: python train_cargo_yolo11.py --epochs 60 --promote")
# Verify files + dynamic dims (no hard-coded W/H)
# NOTE: only assert on the SAM 1 checkpoint when SAM 1 is the ACTIVE segmenter --
# on the SAM 3 path SAM_CHECKPOINT is never created (and must not be downloaded).
if not USE_SAM3:
    assert Path(str(SAM_CHECKPOINT)).exists(), "SAM checkpoint missing"
print("Class names:", list(CARGO_CLASS_NAMES))
print("OpenCV:", cv2.__version__, "| torch:", torch.__version__, "| device:", device)
print(f"Cell 7 ready | segmenter = {'SAM 3 (concept)' if USE_SAM3 else 'SAM 1 ViT-B'} | backend = {SAM_BACKEND}")




In [ ]:
# ============================================================
# CELL 8 — CLEAN CONFIGURATION FOR AUTO-GENERATED BOX TRAINING
# ============================================================
import numpy as np

# All hardcoded coordinate limits have been removed to prevent out-of-bounds bleeding.
ROI_NORMALIZED_BOXES = {
    "LOAD": None,
    "CARGO": None
}

def clip_box(box, image_shape):
    image_h, image_w = image_shape[:2]
    x1, y1, x2, y2 = box

    x1 = int(np.clip(round(x1), 0, image_w))
    y1 = int(np.clip(round(y1), 0, image_h))
    x2 = int(np.clip(round(x2), 0, image_w))
    y2 = int(np.clip(round(y2), 0, image_h))

    if x2 <= x1 or y2 <= y1:
        return None

    return (x1, y1, x2, y2)

def get_roi_boxes(image):
    image_h, image_w = image.shape[:2]
    rois = {"FULL": (0, 0, image_w, image_h)}

    for roi_name in ("LOAD", "CARGO"):
        normalized = ROI_NORMALIZED_BOXES.get(roi_name)
        if normalized is None:
            continue

        x1, y1, x2, y2 = normalized
        box = clip_box(
            (x1 * image_w, y1 * image_h, x2 * image_w, y2 * image_h),
            image.shape
        )

        if box is not None:
            rois[roi_name] = box

    return rois

print("Cell 8: Static coordinates successfully removed. System shifted to dynamic automation.")


In [ ]:
# ============================================================
# CELL 8b — CARGO-SPECIFIC YOLO DETECTION (zip classes) + EXCLUSION ZONES
# ============================================================
# WHAT: custom YOLO11 inference, per-box candidate features (normalized),
#   suspended-load clustering (no merge), dynamic ground rejection.
# NOTE (Colab): run `%pip install -q ultralytics opencv-python` once in CELL 1/8b.
import cv2 as _cv2
import numpy as _np
from ultralytics import YOLO as _YOLO
_t8b = cargo_step("8b. load YOLO detector")
YOLO_MODEL_PATH = str(CARGO_YOLO_WEIGHTS) if "CARGO_YOLO_WEIGHTS" in globals() else "yolo11m.pt"
yolo_model = _YOLO(YOLO_MODEL_PATH)
cargo_done("8b. load YOLO detector", _t8b, weights=YOLO_MODEL_PATH)
MODEL_CLASS_NAMES = list(getattr(yolo_model, "names", {}).values())
USES_ZIP_TAXONOMY = all(n in MODEL_CLASS_NAMES for n in CARGO_CLASS_NAMES)
COCO_CARGO_PROXIES = {"suitcase","backpack","handbag","box","crate","container",
    "refrigerator","toaster","microwave","oven","sink","dining table"}
LIVING_ENTITIES = {"person","dog","cat","horse","bird","cow","sheep","elephant","bear","zebra","giraffe"}
CARGO_CONF_ZIP_TAXONOMY = 0.30; CARGO_CONF_COCO_FALLBACK = 0.15
if USES_ZIP_TAXONOMY:
    TARGET_IDS = list(range(len(MODEL_CLASS_NAMES)))
    print("Cargo zip taxonomy ACTIVE -> targets: Cargo-Box / Wooden-Plate / Cargo-Detection")
else:
    TARGET_IDS = [i for i, n in enumerate(MODEL_CLASS_NAMES) if n not in LIVING_ENTITIES]
    print("Generic COCO taxonomy -> proxies + saliency fallback.")
MIN_CARGO_BOX_PX = 12; MIN_CARGO_AREA_FRACTION = 0.002; PAYLOAD_MIN_ASPECT = 0.10
def detect_hanging_cargo_saliency(image_rgb):
    """Dynamic saliency fallback: edge density + texture in suspended mid-air band."""
    h, w, _ = image_rgb.shape
    min_y, max_y = int(0.10*h), int(0.85*h)
    roi = image_rgb[min_y:max_y, :]
    gray = _cv2.cvtColor(roi, _cv2.COLOR_RGB2GRAY)
    edges = _cv2.Canny(gray, 40, 140)
    dil = _cv2.dilate(edges, _cv2.getStructuringElement(_cv2.MORPH_RECT,(17,17)), iterations=2)
    cnts,_ = _cv2.findContours(dil, _cv2.RETR_EXTERNAL, _cv2.CHAIN_APPROX_SIMPLE)
    solid = (gray < 245).astype(_np.float32)
    cands = []
    for c in cnts:
        a = _cv2.contourArea(c)
        if a < h*w*0.01 or a > h*w*0.70: continue
        x,y,bw,bh = _cv2.boundingRect(c); fy = y+min_y
        if bh/max(1,bw) > 4.5 or bw/max(1,bh) > 4.5: continue
        if (fy+bh) > h*0.92: continue
        if bw > 0.90*w or bh > 0.78*h: continue
        if bw < 0.08*w or bh < 0.05*h: continue
        if bw < 0.25*w and (x <= 0.03*w or (x+bw) >= 0.97*w): continue
        win = solid[y:y+bh, x:x+bw]
        sol = float(win.mean()) if win.size else 0.0
        if sol < 0.35: continue
        yc = (fy+bh/2.0)/h; elev = 1.0-abs(yc-0.55)/0.55
        cands.append([x, fy, x+bw, fy+bh, a*sol*(0.5+0.5*elev)])
    if not cands:
        d = _np.array([[int(0.18*w),int(0.30*h),int(0.82*w),int(0.88*h)]],dtype=_np.float32)
        return d, ["hanging_cargo_payload"], _np.array([0.60],dtype=_np.float32), True
    cands.sort(key=lambda b: b[4], reverse=True); b = cands[0][:4]
    return (_np.array([b],dtype=_np.float32), ["hanging_cargo_payload"],
            _np.array([0.85],dtype=_np.float32), True)

def _extract_detections(res):
    if res is None: return []
    q = None; obb = getattr(res, "obb", None)
    if obb is not None and getattr(obb,"xyxy",None) is not None and len(obb) > 0:
        q = obb.xyxy.detach().cpu().numpy().astype(_np.float32)
    if q is None:
        bx = getattr(res, "boxes", None)
        if bx is None or len(bx) == 0: return []
        q = bx.xyxy.detach().cpu().numpy().astype(_np.float32)
    src = obb if (obb is not None and len(obb) > 0) else getattr(res, "boxes", None)
    if src is None: return []
    cids = src.cls.detach().cpu().numpy().astype(int)
    confs = src.conf.detach().cpu().numpy().astype(_np.float32)
    names = getattr(res, "names", {}) or {}
    out = []
    for b, ci, cf in zip(q, cids, confs):
        nm = names.get(int(ci), str(int(ci))) if isinstance(names,dict) else str(int(ci))
        out.append({"box": _np.asarray(b,dtype=_np.float32),"class_id":int(ci),
                    "class_name":nm,"confidence":float(cf)})
    return out

def _plausible_payload_box(box, fw, fh):
    x1,y1,x2,y2 = [float(v) for v in box]; bw,bh = x2-x1, y2-y1
    if bw < MIN_CARGO_BOX_PX or bh < MIN_CARGO_BOX_PX: return False
    if min(bw,bh) < PAYLOAD_MIN_ASPECT*max(bw,bh): return False
    return (bw*bh) >= MIN_CARGO_AREA_FRACTION*max(1.0, fw*fh)

def _box_iou8(a, b):
    ax1,ay1,ax2,ay2 = [float(v) for v in a]; bx1,by1,bx2,by2 = [float(v) for v in b]
    ix = max(0.0, min(ax2,bx2)-max(ax1,bx1)); iy = max(0.0, min(ay2,by2)-max(ay1,by1))
    inter = ix*iy
    return float(inter/max(1e-6,(ax2-ax1)*(ay2-ay1)+(bx2-bx1)*(by2-by1)-inter))

def candidate_features(box, conf, h, w, others, load_xy=None, excl=None):
    """10 normalized/dynamic candidate features (no hard-coded pixels)."""
    x1,y1,x2,y2 = [float(v) for v in box]; bw,bh = max(1.0,x2-x1), max(1.0,y2-y1)
    cx, cy = (x1+x2)/2.0/w, (y1+y2)/2.0/h
    prox, ov = 1.0, 0.0
    for o in others:
        ox1,oy1,ox2,oy2 = [float(v) for v in o]
        d = _np.hypot((ox1+ox2)/2.0-(x1+x2)/2.0, (oy1+oy2)/2.0-(y1+y2)/2.0)/max(1.0,_np.hypot(w,h))
        prox = min(prox, d); ov = max(ov, _box_iou8(box, o))
    dl = 1.0
    if load_xy is not None:
        dl = float(_np.hypot(cx-load_xy[0], cy-load_xy[1]))
    ex = 0.0
    if excl:
        for e in excl: ex = max(ex, _box_iou8(box, e))
    vpos = 0
    if cy < 0.33: vpos = 0
    elif cy < 0.66: vpos = 1
    else: vpos = 2
    return {"conf":float(conf),"cx":cx,"cy":cy,"aspect":bw/bh,
            "area_ratio":(bw*bh)/max(1.0,w*h),"vpos":vpos,
            "prox_other":prox,"iou_other":ov,"prox_load":dl,
            "dist_bottom":1.0-cy,"excl_overlap":ex}

def cluster_candidates(boxes, h, w, iou_thr=0.30, dist_frac=0.25):
    """Group nearby boxes into suspended-load clusters. NEVER merges boxes."""
    n = len(boxes); parent = list(range(n))
    def find(a):
        while parent[a] != a: parent[a] = parent[parent[a]]; a = parent[a]
        return a
    def union(a, b): parent[find(a)] = find(b)
    for i in range(n):
        for j in range(i+1, n):
            bi, bj = boxes[i], boxes[j]
            ci = ((bi[0]+bi[2])/2.0, (bi[1]+bi[3])/2.0)
            cj = ((bj[0]+bj[2])/2.0, (bj[1]+bj[3])/2.0)
            d = _np.hypot(ci[0]-cj[0], ci[1]-cj[1])/max(1.0,_np.hypot(w,h))
            if _box_iou8(bi,bj) >= iou_thr or d <= dist_frac: union(i,j)
    clusters, mp = [], {}
    for i in range(n):
        r = find(i)
        if r not in mp: mp[r] = len(clusters); clusters.append([])
        clusters[mp[r]].append(i)
    return clusters
def detect_cargo_regions(image_rgb, conf=None, verbose=True):
    """Multi-object detection: one payload entry PER cargo object (never merged)."""
    h, w, _ = image_rgb.shape
    empty = {"payload_boxes":[],"payload_labels":[],"payload_confs":[],"payload_obbs":[],
        "envelope_boxes":[],"envelope_confs":[],"exclusion_boxes":[],"exclusion_labels":[],
        "detections":[],"clusters":[],"load_xy":None,"candidate_feats":[],
        "found_any":False,"source":"NONE"}
    if conf is None:
        conf = CARGO_CONF_ZIP_TAXONOMY if USES_ZIP_TAXONOMY else CARGO_CONF_COCO_FALLBACK
    LOW = 0.10
    def _run_once(thr):
        try:
            res = yolo_model.predict(source=image_rgb, conf=thr, iou=0.45,
                                     classes=TARGET_IDS, device=device, verbose=False)
        except Exception as e:
            if verbose: print(f"  [WARN] YOLO failed: {type(e).__name__}: {e}")
            res = None
        dets = _extract_detections(res[0]) if res else []
        pb, pl, pc, po, eb, ec, xb, xl = [],[],[],[],[],[],[],[]
        for d in dets:
            b = _np.asarray(d["box"],dtype=_np.float32); nm, sc = d["class_name"], d["confidence"]
            x1,y1,x2,y2 = [float(v) for v in b]; bh, bw = y2-y1, x2-x1
            if bh <= 2 or bw <= 2: continue
            if nm not in CARGO_CLASS_TARGETS and (bh > 0.92*h or bw > 0.98*w): continue
            # dynamic ground guard: big box glued to bottom + exclusion context
            if y2 > 0.97*h and bh > 0.30*h and nm not in CARGO_CLASS_TARGETS: continue
            if USES_ZIP_TAXONOMY:
                if nm in CARGO_CLASS_TARGETS:
                    # Keep raw YOLO payload boxes incl. huge ones: on the sacks image the
                    # zip model returns the true load as one big Cargo-Box; the old
                    # height>0.92H drop and downstream scoring discarded it. Size is
                    # handled by scoring, not by dropping detections.
                    pb.append(b);pl.append(nm);pc.append(sc);po.append(None)
                elif nm in CARGO_CLASS_ENVELOPE:
                    # keep envelope even if huge (it is net+load by definition); ground guard still applies above
                    eb.append(b);ec.append(sc)
                elif nm in CARGO_CLASS_EXCLUSIONS: xb.append(b);xl.append(nm)
            elif nm in COCO_CARGO_PROXIES:
                pb.append(b);pl.append("cargo_box_proxy");pc.append(sc);po.append(None)
        keep = list(range(len(pb)))
        # Record (don't drop) implausible flags for debugging; SAM scoring decides.
        implaus = [i for i,b in enumerate(pb) if not _plausible_payload_box(b,w,h)]
        if implaus and verbose and pb:
            print(f"  Note: {len(implaus)} payload box(es) flagged small/sliver (kept for SAM scoring).")
        pb=[pb[i] for i in keep];pl=[pl[i] for i in keep];pc=[pc[i] for i in keep];po=[po[i] for i in keep]
        for b in xb:
            b[0]=float(_np.clip(b[0],0,w));b[2]=float(_np.clip(b[2],0,w))
            b[1]=float(_np.clip(b[1],0,h));b[3]=float(_np.clip(b[3],0,h))
        return pb,pl,pc,po,eb,ec,xb,xl,dets
    pb,pl,pc,po,eb,ec,xb,xl,dets = _run_once(conf)
    if not pb and conf > LOW:
        if verbose: print(f"  No payload at conf={conf:.2f} -> retry at {LOW}.")
        pb,pl,pc,po,eb,ec,xb,xl,dets = _run_once(LOW)
    if not pb and not eb:
        # No payload but envelope/exclusions may still exist (kept above).
        # Only run saliency when the model truly saw nothing at all.
        if dets:
            load_xy = None
            feats = []
            clusters = []
            cids = []
            if verbose:
                print(f"  Detected 0 payload / {len(eb)} envelope / {len(xb)} exclusion")
            return {"payload_boxes":pb,"payload_labels":pl,"payload_confs":pc,"payload_obbs":po,
                "envelope_boxes":eb,"envelope_confs":ec,"exclusion_boxes":xb,"exclusion_labels":xl,
                "detections":dets,"clusters":[],"cluster_ids":[],"load_xy":None,
                "candidate_feats":[],"found_any":bool(eb or xb),
                "source":"ZIP_YOLO" if USES_ZIP_TAXONOMY else "COCO_YOLO"}
        if verbose: print("  Model found 0 targets. Saliency fallback...")
        bx,bl,bc,ok = detect_hanging_cargo_saliency(image_rgb)
        eb=[_np.asarray(b,dtype=_np.float32) for b in bx];ec=[float(c) for c in bc]
        return {**empty,"envelope_boxes":eb,"envelope_confs":ec,"detections":dets,
                "found_any":bool(ok),"source":"SALIENCY_FALLBACK"}
    # suspended-load centroid (dynamic, from payload+envelope, not fixed Y)
    allb = pb+eb
    load_xy = None
    if allb:
        cxs=[(float(b[0])+float(b[2]))/2.0/w for b in allb]
        cys=[(float(b[1])+float(b[3]))/2.0/h for b in allb]
        load_xy=(float(_np.mean(cxs)), float(_np.mean(cys)))
    feats=[candidate_features(b,c,h,w,[o for j,o in enumerate(pb) if j!=i],load_xy,xb)
           for i,(b,c) in enumerate(zip(pb,pc))]
    clusters = cluster_candidates(pb,h,w) if pb else []
    cids=[-1]*len(pb)
    for ci,cl in enumerate(clusters):
        for i in cl: cids[i]=ci
    if verbose:
        print(f"  Detected {len(pb)} payload / {len(eb)} envelope / {len(xb)} exclusion")
    return {"payload_boxes":pb,"payload_labels":pl,"payload_confs":pc,"payload_obbs":po,
        "envelope_boxes":eb,"envelope_confs":ec,"exclusion_boxes":xb,"exclusion_labels":xl,
        "detections":dets,"clusters":clusters,"cluster_ids":cids,"load_xy":load_xy,
        "candidate_feats":feats,"found_any":True,
        "source":"ZIP_YOLO" if USES_ZIP_TAXONOMY else "COCO_YOLO"}

def yolo_detect(image_rgb, conf=None):
    r = detect_cargo_regions(image_rgb, conf=conf, verbose=False)
    boxes=list(r["payload_boxes"])+list(r["envelope_boxes"])
    labels=list(r["payload_labels"])+["hanging_cargo_payload"]*len(r["envelope_boxes"])
    confs=list(r["payload_confs"])+list(r["envelope_confs"])
    if not boxes:
        return _np.empty((0,4),dtype=_np.float32),[],_np.empty((0,),dtype=_np.float32),False
    return _np.asarray(boxes,dtype=_np.float32),labels,_np.asarray(confs,dtype=_np.float32),True

print("Cell 8b ready: multi-object YOLO | taxonomy=",
      "zip" if USES_ZIP_TAXONOMY else "coco",
      "| OBB=","yes" if getattr(yolo_model,"task","")=="obb" else "no")





In [ ]:
# ============================================================
# CELL 8c — DYNAMIC LOAD REGION ESTIMATOR & CARGO PROMPT ENGINE (SAM 3 + SAM 1)
# ============================================================
# WHAT: per-candidate dynamic ROI (W/H-proportional expansion) + SAM prompts.
#   SAM 3 path  -> CONCEPT TEXT prompts + positive/negative BOX prompts.
#   SAM 1 path  -> box + pos/neg points (legacy, kept for the fallback).
# No manual prompts, no fixed coordinates. One prompt PER cargo object.

# ---------------------------------------------------------------------------
# 8c-SAM3. CONCEPT TEXT PROMPTS  (the detection-rate lever)
# ---------------------------------------------------------------------------
# SAM 3 grounds natural-language CONCEPTS, not just geometry. A single phrasing
# caps recall at whatever synonyms that phrasing happens to contain, so the
# recall ceiling is raised by querying several phrasings and UNIONING the masks.
# Each entry is tried; a phrasing that returns nothing costs one forward pass and
# loses nothing. Ordered most->least specific so the printed progress reads
# sensibly.
CARGO_CONCEPT_PROMPTS = (
    "suspended cargo load hanging in the air",
    "cargo inside a cargo net",
    "a pallet of goods and freight",
    "a bundle of sacks and bags",
    "wooden crate of cargo",
)
#: Cap on phrasings per image. Each extra phrasing is one extra grounding pass,
#: so this trades recall against runtime. 3 is the sweet spot for this dataset.
SAM3_MAX_TEXT_PROMPTS = 3
#: Minimum area fraction (of the whole frame) for a concept hit to be trusted.
#: Guards against SAM 3 grounding one word and painting half the sky.
SAM3_MIN_AREA_FRACTION = 0.004
#: Reject a concept hit that swallows more than this fraction of the frame --
#: that is rigging/sky, not a discrete cargo object.
SAM3_MAX_AREA_FRACTION = 0.70

def build_sam3_prompts(payload_boxes, envelope_boxes, exclusion_boxes, image_rgb,
                       text_prompts=CARGO_CONCEPT_PROMPTS):
    """SAM 3 prompt bundle: concept text + normalized [cx,cy,w,h] pos/neg boxes.

    Negative boxes come from the detected Net/Strap/Chain/Crane/Hook classes.
    The SAM 1 pipeline could only subtract those AFTER masking, which erodes the
    cargo it was trying to keep; handing them to SAM 3 as `label=False` prompts
    suppresses the rigging at the source instead.
    """
    h, w, _ = image_rgb.shape
    texts = list(text_prompts)[:max(1, int(SAM3_MAX_TEXT_PROMPTS))]
    pos_boxes = []
    for b in list(payload_boxes) + list(envelope_boxes):
        x1, y1, x2, y2 = [float(v) for v in b]
        bw, bh = max(1.0, x2 - x1), max(1.0, y2 - y1)
        pos_boxes.append([(x1 + bw / 2.0) / w, (y1 + bh / 2.0) / h, bw / w, bh / h])
    # SAM 3 expects NORMALIZED [cx,cy,w,h]; clamp so a box that runs off-frame
    # (YOLO OBB can overshoot) cannot produce an out-of-range prompt.
    pos_boxes = [[min(max(0.0, v), 1.0) for v in bx] for bx in pos_boxes]
    neg_boxes = []
    for b in exclusion_boxes:
        x1, y1, x2, y2 = [float(v) for v in b]
        bw, bh = max(1.0, x2 - x1), max(1.0, y2 - y1)
        neg_boxes.append([(x1 + bw / 2.0) / w, (y1 + bh / 2.0) / h, bw / w, bh / h])
    neg_boxes = [[min(max(0.0, v), 1.0) for v in bx] for bx in neg_boxes]
    return {"texts": texts, "pos_boxes": pos_boxes, "neg_boxes": neg_boxes,
            "shape": (h, w)}

import numpy as _np8c
import cv2 as _cv8c
USE_MANUAL_OVERRIDE = False
MANUAL_PROMPTS = {}
def get_manual_target_boxes(image_index, roi_shape, roi_name="FULL"):
    return []  # manual override permanently disabled (input independence)

def estimate_cargo_load_region(net_box, image_rgb):
    """Density fallback when only the net envelope is known. Scene-adaptive."""
    x1,y1,x2,y2 = [int(v) for v in net_box]; h,w,_ = image_rgb.shape
    x1c,y1c,x2c,y2c = max(0,min(w,x1)),max(0,min(h,y1)),max(0,min(w,x2)),max(0,min(h,y2))
    nw, nh = x2c-x1c, y2c-y1c
    if nw <= 1 or nh <= 1:
        return (_np8c.array([x1c,y1c,x2c,y2c],dtype=_np8c.float32),
                [[float((x1c+x2c)/2),float((y1c+y2c)/2)]],[],y1c)
    gray = _cv8c.cvtColor(image_rgb,_cv8c.COLOR_RGB2GRAY)
    cx1, cx2 = max(0,int(x1c+nw*0.15)), min(w,int(x2c-nw*0.15))
    ring = _np8c.concatenate([gray[0:4,:].ravel(),gray[-4:,:].ravel(),
        gray[:,0:4].ravel(),gray[:,-4:].ravel()]).astype(_np8c.float32)
    bg = float(_np8c.median(ring))
    crop = gray[y1c:y2c,cx1:cx2].astype(_np8c.float32)
    gx = _cv8c.Sobel(crop,_cv8c.CV_32F,1,0,ksize=3); gy = _cv8c.Sobel(crop,_cv8c.CV_32F,0,1,ksize=3)
    grad = _cv8c.magnitude(gx,gy)
    nonbg = ((_np8c.abs(crop-bg)>25.0)|(grad>40.0)).astype(_np8c.uint8)
    rows = _np8c.sum(nonbg,axis=1)
    if len(rows)>0 and _np8c.max(rows)>0:
        th = _np8c.max(rows)*0.30; solid = _np8c.where(rows>th)[0]
        if len(solid)>0: ys,ye = y1c+_np8c.min(solid), y1c+_np8c.max(solid)
        else: ys,ye = y1c+int(nh*0.50), y2c
    else: ys,ye = y1c+int(nh*0.50), y2c
    if (ye-ys) > int(0.65*h): ys = max(int(ys), int(ye)-int(0.65*h))
    box = _np8c.array([x1c,ys,x2c,ye],dtype=_np8c.float32)
    pos = [[x1c+nw*0.50,ys+(ye-ys)*0.50],[x1c+nw*0.35,ys+(ye-ys)*0.50],[x1c+nw*0.65,ys+(ye-ys)*0.50]]
    neg = [[x1c+nw*0.50,max(0,y1c+(ys-y1c)*0.35)]]
    return box,pos,neg,int(ys)

ROI_EXPAND_FRAC = 0.15  # fraction of W/H — proportional, never fixed pixels
def expand_roi(box, h, w, frac=ROI_EXPAND_FRAC):
    x1,y1,x2,y2=[float(v) for v in box]; bw,bh=max(1.0,x2-x1),max(1.0,y2-y1)
    px,pY=bw*frac,bh*frac
    return _np8c.array([max(0.0,x1-px),max(0.0,y1-pY),min(float(w),x2+px),min(float(h),y2+pY)],dtype=_np8c.float32)

def build_cargo_prompts(payload_boxes, envelope_boxes, exclusion_boxes, image_rgb):
    """LEGACY single-prompt view: union box (kept for Cell 9 fallback compat)."""
    h,w,_ = image_rgb.shape
    boxes=[_np8c.asarray(b,dtype=_np8c.float32) for b in payload_boxes]
    if boxes:
        x1=min(float(b[0]) for b in boxes);y1=min(float(b[1]) for b in boxes)
        x2=max(float(b[2]) for b in boxes);y2=max(float(b[3]) for b in boxes)
        pb=_np8c.array([x1,y1,x2,y2],dtype=_np8c.float32)
        pos=[[float((b[0]+b[2])/2.0),float((b[1]+b[3])/2.0)] for b in boxes]
        pos+= [[float((x1+x2)/2.0),float(y1+(y2-y1)*0.35)],[float((x1+x2)/2.0),float(y1+(y2-y1)*0.70)]]
        ys,src=int(max(0.0,y1)),"ZIP_CARGO_BOX"
    elif envelope_boxes:
        env=_np8c.asarray(envelope_boxes[0],dtype=_np8c.float32)
        if len(envelope_boxes)>1:
            env=_np8c.array([min(float(b[0]) for b in envelope_boxes),min(float(b[1]) for b in envelope_boxes),
                             max(float(b[2]) for b in envelope_boxes),max(float(b[3]) for b in envelope_boxes)],dtype=_np8c.float32)
        pb,pos,_,ys=estimate_cargo_load_region(env,image_rgb)
        pos=[list(map(float,p)) for p in pos];src="DENSITY_ENVELOPE"
    else:
        pb=_np8c.array([w*0.10,h*0.30,w*0.90,h*0.95],dtype=_np8c.float32)
        ys,pos,src=int(h*0.45),[[w*0.50,h*0.65]],"GLOBAL_FALLBACK"
    neg=[]
    for b in exclusion_boxes:
        a,b1,c,d=[float(v) for v in b]
        if d<=ys+1: neg.append([(a+c)/2.0,max(0.0,(b1+d)/2.0)])
    span=max(1.0,float(pb[3]-pb[1]));bw=max(1.0,float(pb[2]-pb[0]))
    neg+= [[float(pb[0]+bw*0.50),float(max(0.0,pb[1]-span*0.35))],
           [float(pb[0]+bw*0.15),float(max(0.0,pb[1]-span*0.20))]]
    cl=lambda p:[float(_np8c.clip(p[0],0,w-1)),float(_np8c.clip(p[1],0,h-1))]
    return pb,[cl(p) for p in pos],[cl(p) for p in neg],int(ys),src
def build_per_object_prompts(payload_boxes, envelope_boxes, exclusion_boxes, image_rgb):
    """PRIMARY multi-object path: one dynamic ROI + box/pos/neg per cargo object."""
    h,w,_ = image_rgb.shape
    out=[]
    if payload_boxes:
        # estimate shared load band from exclusion geometry (dynamic, not fixed Y)
        ys_ref = h
        for b in exclusion_boxes:
            try: ys_ref=min(ys_ref,float(b[3]))
            except Exception: pass
        for i,b in enumerate(payload_boxes):
            b=_np8c.asarray(b,dtype=_np8c.float32)
            roi=expand_roi(b,h,w)
            cx,cy=float((b[0]+b[2])/2.0),float((b[1]+b[3])/2.0)
            bw,bh=float(b[2]-b[0]),float(b[3]-b[1])
            pos=[[cx,cy],[float(b[0]+bw*0.35),float(b[1]+bh*0.35)],[float(b[0]+bw*0.65),float(b[1]+bh*0.70)]]
            neg=[]
            for e in exclusion_boxes:
                a,b1,c,d=[float(v) for v in e]
                if d<=float(b[1])+1: neg.append([(a+c)/2.0,max(0.0,(b1+d)/2.0)])
            span=max(1.0,float(b[3]-b[1]));wb=max(1.0,float(b[2]-b[0]))
            neg+= [[float(b[0]+wb*0.50),float(max(0.0,b[1]-span*0.35))],
                   [float(b[0]+wb*0.15),float(max(0.0,b[1]-span*0.20))]]
            cl=lambda p:[float(_np8c.clip(p[0],0,w-1)),float(_np8c.clip(p[1],0,h-1))]
            out.append({"index":i,"yolo_box":_np8c.asarray(b,dtype=_np8c.float32),
                        "roi":roi,"box":_np8c.asarray(b,dtype=_np8c.float32),
                        "pos":[cl(p) for p in pos],"neg":[cl(p) for p in neg],
                        "y_start":int(max(0.0,float(b[1]))),"source":"PER_OBJECT_YOLO"})
        return out
    # fallback: single envelope-derived prompt
    pb,pos,neg,ys,src=build_cargo_prompts([],envelope_boxes,exclusion_boxes,image_rgb)
    roi=expand_roi(pb,h,w)
    return [{"index":0,"yolo_box":_np8c.asarray(pb,dtype=_np8c.float32),"roi":roi,
             "box":_np8c.asarray(pb,dtype=_np8c.float32),"pos":pos,"neg":neg,
             "y_start":int(ys),"source":src}]
print(f"Cell 8c ready: per-object dynamic ROI + prompts (multi-object) | "
      f"SAM 3 concepts: {len(CARGO_CONCEPT_PROMPTS)} phrasings "
      f"(using {SAM3_MAX_TEXT_PROMPTS}/image).")



In [ ]:
# ============================================================
# CELL 9 — TIGHT CARGO BOUNDING BOX & EXCLUSION-AWARE MASK FILTER
# ============================================================
# WHAT: per-object crop/upscale -> SAM multimask -> normalized weighted select
#   -> net rejection -> refine -> tight box from FINAL mask (never reuse YOLO box).
import time as _t9
import cv2 as _cv9
import numpy as _np9
FALLBACK_POINTS_PER_SIDE=12; FALLBACK_PRED_IOU_THRESH=0.82
FALLBACK_STABILITY_THRESH=0.88; FALLBACK_MIN_MASK_REGION_AREA=500
MASK_BG_CUTOFF=242
RANK_W_SAM,RANK_W_IOU,RANK_W_FILL,RANK_W_EXCLUSION=5.0,4.0,2.0,3.0
OBB_TIGHTNESS_RATIO=0.92; CONTAINMENT_PAD=0.12
cargo_obb_by_image=[]
# Score weights (normalized terms): meaning in comments at rank function.
W_YOLO=1.0; W_SAM=1.0; W_FILL=1.0; W_OVERLAP=1.5; W_CARGO=1.0; W_SHAPE=0.8
W_SPATIAL=0.6; P_NET=2.5; P_GROUND=2.0; P_BG=1.5; P_BORDER=2.0
def boxes_to_mask(boxes, shape):
    m=_np9.zeros(shape[:2],dtype=bool); hh,ww=shape[:2]
    for b in boxes:
        x1,y1,x2,y2=[int(round(float(v))) for v in b]
        x1,x2=max(0,min(ww,x1)),max(0,min(ww,x2));y1,y2=max(0,min(hh,y1)),max(0,min(hh,y2))
        if x2>x1 and y2>y1: m[y1:y2,x1:x2]=True
    return m
def box_iou(a,b):
    ax1,ay1,ax2,ay2=[float(v) for v in a];bx1,by1,bx2,by2=[float(v) for v in b]
    ix,iy=max(ax1,bx1),max(ay1,by1);jx,jy=min(ax2,bx2),min(ay2,by2)
    inter=max(0.0,jx-ix)*max(0.0,jy-iy)
    return float(inter/max(1e-6,(ax2-ax1)*(ay2-ay1)+(bx2-bx1)*(by2-by1)-inter))
def scene_background_mask(img, color_tol=40.0, smooth_max_std=12.0):
    h,w=img.shape[:2]; f=img.astype(_np9.float32)
    ring=_np9.concatenate([f[0:4,:,:].reshape(-1,3),f[-4:,:,:].reshape(-1,3),
        f[:,0:4,:].reshape(-1,3),f[:,-4:,:].reshape(-1,3)])
    bg=_np9.median(ring,axis=0)
    cm=_np9.linalg.norm(f-bg,axis=2)<color_tol
    g=_cv9.cvtColor(img,_cv9.COLOR_RGB2GRAY).astype(_np9.float32)
    mn=_cv9.boxFilter(g,-1,(7,7));ms=_cv9.boxFilter(g*g,-1,(7,7))
    std=_np9.sqrt(_np9.maximum(ms-mn*mn,0.0))
    cand=((cm)&(std<smooth_max_std)).astype(_np9.uint8)
    nl,lb=_cv9.connectedComponents(cand,connectivity=8)
    bl=_np9.unique(_np9.concatenate([lb[0,:],lb[-1,:],lb[:,0],lb[:,-1]]));bl=bl[bl!=0]
    if bl.size==0: return _np9.zeros((h,w),dtype=bool)
    return _np9.isin(lb,bl)
def mask_geometry(mask):
    m=(mask>0).astype(_np9.uint8); area=float(m.sum())
    ys,xs=_np9.where(m>0)
    if len(xs)==0:
        return {"area":0.0,"fill":0.0,"compact":0.0,"solid":0.0,"extent":0.0,
                "aspect":1.0,"border":0.0,"box":_np9.array([0,0,0,0],dtype=_np9.float32),"ncomp":0}
    x1, x2 = xs.min(), xs.max(); y1, y2 = ys.min(), ys.max()
    bw,bh=max(1.0,x2-x1+1),max(1.0,y2-y1+1)
    cnts,_=_cv9.findContours(m,_cv9.RETR_EXTERNAL,_cv9.CHAIN_APPROX_SIMPLE)
    per=0.0; hull_a=0.0
    if cnts:
        c=max(cnts,key=_cv9.contourArea); per=float(_cv9.arcLength(c,True))
        hull=_cv9.convexHull(c); hull_a=float(_cv9.contourArea(hull))
    compact=float(4*_np9.pi*area/max(1.0,per*per)) if per>0 else 0.0
    solid=float(area/max(1.0,hull_a)) if hull_a>0 else 1.0
    h,w=m.shape
    ring=_np9.zeros_like(m,dtype=bool);ring[:3,:]=True;ring[-3:,:]=True;ring[:,:3]=True;ring[:,-3:]=True
    border=float((m.astype(bool)&ring).sum())/max(1.0,float(ring.sum()))
    nl,_,_,_= _cv9.connectedComponentsWithStats(m)
    return {"area":area,"fill":min(1.0,area/max(1.0,bw*bh)),"compact":min(1.0,compact),
            "solid":min(1.0,solid),"extent":min(1.0,area/max(1.0,bw*bh)),
            "aspect":bw/max(1.0,bh),"border":border,
            "box":_np9.array([x1,y1,x2,y2],dtype=_np9.float32),"ncomp":int(nl-1)}

def thin_net_penalty(mask):
    """High when mask is dominated by thin elongated net/strap lines."""
    m=(mask>0).astype(_np9.uint8)
    if m.sum()==0: return 0.0
    skel=_cv9.ximgproc.thinning(m*255) if hasattr(_cv9,'ximgproc') else m
    skel_px=float((skel>0).sum()); area=float(m.sum())
    thin_ratio=skel_px/max(1.0,area)
    k=_cv9.getStructuringElement(_cv9.MORPH_RECT,(7,1))
    hbar=_cv9.morphologyEx(m,_cv9.MORPH_OPEN,k).sum()/max(1.0,area)
    k2=_cv9.getStructuringElement(_cv9.MORPH_RECT,(1,7))
    vbar=_cv9.morphologyEx(m,_cv9.MORPH_OPEN,k2).sum()/max(1.0,area)
    elong=max(hbar,vbar)
    return float(min(1.0, 0.6*min(1.0,thin_ratio*8.0)+0.4*min(1.0,elong*3.0)))

def rank_sam_candidates(masks, sam_scores, yolo_box, yolo_conf, roi_img, excl_mask, y_start):
    """Normalized weighted score per SAM mask. Returns (best_idx, best_score, details)."""
    tb=_np9.asarray(yolo_box,dtype=_np9.float32)
    H,W=roi_img.shape[:2]; img_area=max(1.0,H*W)
    rows=[]
    for i,(m,s) in enumerate(zip(masks,sam_scores)):
        g=mask_geometry(m); area=g["area"]
        if area<=0: continue
        fill=g["fill"]; overlap=box_iou(g["box"],tb)
        cargo_cons=min(1.0,overlap*0.7+fill*0.3)
        shape=0.5*g["compact"]+0.5*g["solid"]
        cy=(float(g["box"][1])+float(g["box"][3]))/2.0/H
        spatial=max(0.0,1.0-abs(cy-0.62)/0.62)
        bleed=0.0
        if excl_mask is not None and excl_mask.any():
            exf=float(excl_mask.sum())/max(1.0,H*W)
            raw_bleed=float(_np9.logical_and(m.astype(bool),excl_mask).sum())/max(1.0,area)
            # A HUGE exclusion box (net envelope covering most of the ROI) makes raw
            # bleed meaningless — every mask overlaps it. Discount bleed by the
            # exclusion's own size (dynamic, image-relative): bleed only counts when
            # the exclusion is a LOCALIZED strap/rope, not the whole net.
            bleed=raw_bleed*max(0.0,1.0-exf*2.0)
        net_pen=max(bleed,thin_net_penalty(m))
        touches_bottom=float(g["box"][3])>=H-3
        ground_pen=1.0 if touches_bottom and g["area"]/img_area>0.25 else (0.5 if touches_bottom else 0.0)
        bg=_np9.mean((_cv9.cvtColor(roi_img,_cv9.COLOR_RGB2GRAY)<MASK_BG_CUTOFF).astype(_np9.float32))
        bg_pen=max(0.0,1.0-bg)*0.3
        border_pen=min(1.0,g["border"]*8.0)
        ncomp_pen=min(1.0,max(0,g["ncomp"]-1)/4.0)
        score=(W_YOLO*min(1.0,float(yolo_conf))+W_SAM*float(s)+W_FILL*fill
               +W_OVERLAP*overlap+W_CARGO*cargo_cons+W_SHAPE*shape+W_SPATIAL*spatial
               -P_NET*net_pen-P_GROUND*ground_pen-P_BG*bg_pen-P_BORDER*border_pen-0.5*ncomp_pen)
        rows.append((score,i,float(s),fill,overlap,net_pen,ground_pen,border_pen,g))
    if not rows: return None,None,[]
    # rank by fill*overlap (cargo solidity) pooled with confidence: big solid
    # loads (sacks) beat small bright bands; tiny bright bands lose even if SAM
    # loves them. Terms are image-relative; no fixed pixels.
    best_key = None; bi = None; bs = 0.0
    for (score,i,ss,fill,overlap,npen,gpen,bpen,g) in rows:
        key = (fill*overlap+1e-6)*(0.5+0.5*min(1.0,float(ss)))*(0.5+0.5*min(1.0,float(yolo_conf)))
        if best_key is None or key > best_key: best_key = key; bi = i; bs = score
    return bi, bs, rows
def clean_cargo_mask(raw_mask, roi_image, target_box, exclusion_mask, info=None):
    clean=(raw_mask.astype(bool)).astype(_np9.uint8)
    if exclusion_mask is not None and _np9.asarray(exclusion_mask).any():
        ex=_np9.asarray(exclusion_mask).astype(bool)
        # Only subtract LOCALIZED exclusions (straps/ropes). A huge net envelope
        # covering most of the ROI must NOT wipe the cargo: skip subtraction when
        # the exclusion covers >35% of the ROI (dynamic, image-relative).
        if float(ex.sum())/max(1.0,ex.size)>0.35:
            pass
        else:
            # Dynamic exclusion shrink: mask pixels merely TOUCHING a net/strap box
            # should not be wiped with it. The radius is image-relative (~2% of the
            # ROI diagonal) but CAPPED at 10% of the exclusion's own smaller side,
            # otherwise a thin strap/rope is eroded out of existence and localized
            # exclusions would never be subtracted at all.
            dd=max(1,int(0.02*float(_np9.hypot(*clean.shape[:2]))))
            ys,xs=_np9.nonzero(ex)
            if xs.size:
                thin_side=max(1.0,float(min(xs.max()-xs.min()+1,ys.max()-ys.min()+1)))
                dd=int(min(dd,max(1,int(0.10*thin_side))))
            exk=_cv9.getStructuringElement(_cv9.MORPH_ELLIPSE,(2*dd+1,2*dd+1))
            ex=_cv9.erode(ex.astype(_np9.uint8),exk,iterations=1).astype(bool)
            clean[ex]=0
    if target_box is not None:
        _,by1,_,by2=[float(v) for v in target_box]; bh=max(1.0,by2-by1)
        # keep the load band DYNAMIC: relative to the prompt box, never fixed rows.
        top=int(max(0,by1-bh*0.60));bot=int(min(clean.shape[0],by2+bh*0.25))
        band=_np9.zeros_like(clean);band[top:bot,:]=1;clean=_cv9.bitwise_and(clean,band)
    if target_box is not None:
        bx1,by1,bx2,by2=[float(v) for v in target_box]
        # containment is generous for far/small cargo: pad scales with 1/box-size.
        bw9,bh9=max(1.0,bx2-bx1),max(1.0,by2-by1)
        dyn_pad=min(0.60,CONTAINMENT_PAD+0.25*min(1.0,120.0/max(bw9,bh9)))
        px=max(4.0,(bx2-bx1)*dyn_pad);pY=max(4.0,(by2-by1)*dyn_pad)
        win=_np9.zeros_like(clean)
        a,b=int(max(0,_np9.floor(bx1-px))),int(min(clean.shape[1],_np9.ceil(bx2+px)))
        c,d=int(max(0,_np9.floor(by1-pY))),int(min(clean.shape[0],_np9.ceil(by2+pY)))
        if b>a and d>c: win[c:d,a:b]=1;clean=_cv9.bitwise_and(clean,win)
    clean[scene_background_mask(roi_image)]=0
    nl,lb,stats,_=_cv9.connectedComponentsWithStats(clean)
    if nl>1:
        big=1+int(_np9.argmax(stats[1:,_cv9.CC_STAT_AREA]))
        keep=(lb==big).astype(_np9.uint8)
        # keep secondary lobes (>=8% of largest, dynamic): sacks/slings split into
        # 2-3 blobs; the old 15% cut dropped real cargo on far/small loads.
        for li in range(1,nl):
            if li!=big and float(stats[li,_cv9.CC_STAT_AREA])>=0.08*float(stats[big,_cv9.CC_STAT_AREA]):
                keep[lb==li]=1
        # merge-close split sack lobes: dilate by ~1.5% ROI diag then fill the hull
        # so the tight box spans the full load (dynamic, image-relative).
        # CAP the merge: hull may add at most 60% new pixels vs kept lobes, else a
        # stray speck would balloon the box (this is what broke img1: 203->300px).
        dd2=max(1,int(0.015*float(_np9.hypot(*clean.shape[:2]))))
        kk=_cv9.getStructuringElement(_cv9.MORPH_ELLIPSE,(2*dd2+1,2*dd2+1))
        mg=_cv9.dilate(keep,kk,iterations=1)
        cn2,_=_cv9.findContours(mg,_cv9.RETR_EXTERNAL,_cv9.CHAIN_APPROX_SIMPLE)
        if cn2:
            c2=max(cn2,key=_cv9.contourArea)
            f2=_np9.zeros_like(keep);_cv9.drawContours(f2,[c2],-1,1,thickness=-1)
            # CLOSE (erode back by the same dd2): bridging the inter-sack gap must NOT
            # grow the bounding box by the dilation margin, the box must stay the real
            # cargo extent (dynamic, image-relative).
            f2=_cv9.erode(f2,kk,iterations=1)
            new_px=float(f2.sum())-float(keep.sum())
            if new_px<=0.60*max(1.0,float(keep.sum())):
                keep=_np9.maximum(keep,f2)
        _ys,_xs=_np9.where(keep>0)
        _spw=(_xs.max()-_xs.min()+1)/max(1.0,float(clean.shape[1])) if _xs.size else 0.0
        _sph=(_ys.max()-_ys.min()+1)/max(1.0,float(clean.shape[0])) if _ys.size else 0.0
        # NON-DISCRIMINATIVE MASK GUARD: a SAM winner that swallows >=60% of the WHOLE ROI
        # and spans >=85% of it on BOTH axes is not localising the load — it glued the net
        # and the ROI padding onto the cargo, so the ROI padding would inflate the tight
        # box. Trust the detector box then (padded by a hair, 2% of its own size). Every
        # term here is a fraction of the live ROI, no fixed pixels/colours.
        _mfrac=float(keep.sum())/max(1.0,float(keep.size))
        if target_box is not None and _spw>=0.85 and _sph>=0.85 and _mfrac>=0.60:
            bx9,by9,bz9,bw9=[float(v) for v in target_box]
            qx=max(1.0,0.02*max(1.0,bz9-bx9));qy=max(1.0,0.02*max(1.0,bw9-by9))
            win=_np9.zeros_like(keep)
            a,b=int(max(0,_np9.floor(bx9-qx))),int(min(clean.shape[1],_np9.ceil(bz9+qx)))
            c,d=int(max(0,_np9.floor(by9-qy))),int(min(clean.shape[0],_np9.ceil(bw9+qy)))
            if b>a and d>c:
                win[c:d,a:b]=1;keep=_cv9.bitwise_and(keep,win)
            if isinstance(info,dict): info["non_discriminative"]=True
            print(f"  [lobes] non-discriminative mask (span {_spw:.2f}/{_sph:.2f}, "
                  f"fill {_mfrac:.2f}) -> clamped to detector box")
        clean=keep
    elif clean.sum()==0: return clean
    clean=_cv9.morphologyEx(clean,_cv9.MORPH_CLOSE,
        _cv9.getStructuringElement(_cv9.MORPH_ELLIPSE,(5,5)),iterations=1)
    cnts,_=_cv9.findContours(clean,_cv9.RETR_EXTERNAL,_cv9.CHAIN_APPROX_SIMPLE)
    if cnts:
        f=_np9.zeros_like(clean);_cv9.drawContours(f,cnts,-1,1,thickness=-1);clean=f
    return clean

def get_roi_boxes(image_rgb):
    h,w=image_rgb.shape[:2]
    return {"FULL": (0,0,w,h)}

# ---------------------------------------------------------------------------
# 9-SAM3. CONCEPT-PATH SEGMENTATION
# ---------------------------------------------------------------------------
# Execution shape differs fundamentally from SAM 1: SAM 3 encodes the WHOLE image
# ONCE (the expensive part), then answers many prompts against that one embedding.
# SAM 1 re-encoded a crop per object. So this path is both the higher-recall path
# AND the faster one.
#
# Two SAM 3 API constraints drive the structure (sam3_image_processor.py):
#   * set_text_prompt() ERASES the previous text prompt            (:121)
#   * add_geometric_prompt() APPENDS to state["geometric_prompt"] (:151)
# => reset_all_prompts(state) before every independent prompt attempt, otherwise
#    boxes leak across objects and results contaminate each other.

def _sam3_union_instances(states, image_shape, min_area_frac, max_area_frac):
    """Merge instances from several concept phrasings into one candidate list.

    Unioning across phrasings is what actually raises recall: a phrasing that
    misses a given load contributes nothing there and another phrasing covers it.
    Overlapping instances are de-duplicated by mask overlap.
    """
    H,W=image_shape[:2]; frame=float(max(1,H*W))
    out=[]
    for st in states:
        masks=st.get("masks"); boxes=st.get("boxes"); scores=st.get("scores")
        if masks is None or len(masks)==0: continue
        masks=_np9.asarray(masks)
        if masks.ndim==2: masks=masks[None,...]
        boxes=boxes if boxes is not None and len(boxes) else [None]*len(masks)
        scores=(_np9.asarray(scores,dtype=_np9.float32).reshape(-1)
                if scores is not None else _np9.ones(len(masks),dtype=_np9.float32))
        for i in range(len(masks)):
            m=(masks[i]>0); a=float(m.sum())
            if a<=0: continue
            frac=a/frame
            if frac<min_area_frac or frac>max_area_frac: continue
            sc=float(scores[i]) if i<len(scores) else 0.5
            dup=False
            for prev in out:                      # keep the stronger of two overlaps
                inter=float(_np9.logical_and(m,prev["mask"]).sum())
                if inter/max(1.0,a) >= 0.60:
                    dup=True
                    if sc>prev["score"]: prev["score"]=sc
                    break
            if dup: continue
            bx=boxes[i] if i<len(boxes) else None
            out.append({"mask":m,"score":sc,
                        "box":_np9.asarray(bx,dtype=_np9.float32) if bx is not None else None})
    return out

def candidate_selection_score_probe(mask, box, label, conf, sam_score, shape):
    """Mirror of Cell 10's candidate_selection_score, inlined so Cell 9 can pre-rank
    without importing Cell 10 (which runs later). Same terms, so a SAM 3 object is
    accepted or rejected on exactly the same grounds as a SAM 1 one."""
    H,W=shape[:2]; A=float(H*W); ma=float(mask.sum())
    if ma<=0: return -500.0
    x1,y1,x2,y2=[float(v) for v in box]
    ys,xs=_np9.where(mask)
    mba=float(max(1,xs.max()-xs.min()+1)*max(1,ys.max()-ys.min()+1)) if len(ys) else max(1.0,(x2-x1)*(y2-y1))
    fill=min(1.0,ma/max(1.0,mba)); ar=min(1.0,ma/max(1.0,A))
    area=max(0.0,1.0-abs(ar-0.25)/0.75)
    if ar<0.03: area=0.0
    over=8.0*max(0.0,ar-0.40); under=3.0*max(0.0,(0.03-ar)/0.03)
    floor=10.0 if y2>=(H-5) else 0.0; top=2.0 if y1<=2 else 0.0
    r=_np9.zeros(mask.shape,dtype=bool); r[:3,:]=True; r[-3:,:]=True; r[:,:3]=True; r[:,-3:]=True
    bord=6.0*max(0.0,float((mask&r).sum())/max(1.0,float(r.sum()))-0.05)
    fb=2.0 if label=="cargo_payload" else 0.0
    sky=6.0 if (y1<=3 and fill<0.45) else 0.0
    return float(8.0*float(sam_score)+6.0*fill+4.0*float(conf)+2.0*area-floor-top-over-under-bord-fb-sky)

def segment_image_sam3(image_rgb, payload_boxes, envelope_boxes, exclusion_boxes,
                       labels, confs, image_index):
    """Run SAM 3 with concept text + pos/neg boxes. Emits Cell-10 tuples directly."""
    H,W=image_rgb.shape[:2]
    t_img=_t9.perf_counter()
    bundle=build_sam3_prompts(payload_boxes,envelope_boxes,exclusion_boxes,image_rgb)
    proc=sam3_processor
    state=proc.set_image(image_rgb)          # ONE encode for the whole image
    cargo_done(f"    SAM3 encode [image {image_index+1}]",t_img,shape=f"{W}x{H}")

    collected=[]
    # ---- pass A: concept text only -- open-vocabulary recall, no YOLO needed ----
    for text in bundle["texts"]:
        t_t=_t9.perf_counter()
        try:
            proc.reset_all_prompts(state)
            st=proc.set_text_prompt(prompt=text,state=state)
            got=_sam3_union_instances([st],(H,W),SAM3_MIN_AREA_FRACTION,SAM3_MAX_AREA_FRACTION)
            collected.extend(got)
            cargo_done(f"    SAM3 concept [image {image_index+1}]",t_t,
                       found=len(got),prompt=text[:28])
        except Exception as e:
            print(f"  [WARN] SAM3 text prompt failed ({text[:24]}): {type(e).__name__}: {e}")

    # ---- pass B: concept text + positive/negative boxes -- the precision pass ----
    # YOLO boxes steer SAM 3 onto the detected load, and the Net/Strap/Chain/
    # Crane/Hook exclusions act as TRUE negative prompts, suppressing rigging at the
    # source rather than eroding the finished mask afterwards (which is all the
    # SAM 1 pipeline could do, and is what destroyed thin cargo).
    if bundle["pos_boxes"] or bundle["neg_boxes"]:
        t_b=_t9.perf_counter()
        try:
            for text in bundle["texts"]:
                proc.reset_all_prompts(state)
                st=proc.set_text_prompt(prompt=text,state=state)
                for nb in bundle["neg_boxes"]:
                    st=proc.add_geometric_prompt(state=st,box=nb,label=False)
                for pb in bundle["pos_boxes"]:
                    st=proc.add_geometric_prompt(state=st,box=pb,label=True)
                got=_sam3_union_instances([st],(H,W),SAM3_MIN_AREA_FRACTION,SAM3_MAX_AREA_FRACTION)
                for g in got: g["score"]=min(1.0,float(g["score"])*1.15)  # guided > free
                collected.extend(got)
            cargo_done(f"    SAM3 box-guided [image {image_index+1}]",t_b,
                       hits=len(collected),pos=len(bundle["pos_boxes"]),neg=len(bundle["neg_boxes"]))
        except Exception as e:
            print(f"  [WARN] SAM3 box prompting failed: {type(e).__name__}: {e}")

    if not collected:
        print(f"  SAM3: no cargo above conf>{SAM3_CONF_THRESHOLD} for these concepts.")
        return [],[]

    # De-duplicate ACROSS passes. Each phrasing calls _sam3_union_instances with a
    # fresh list, so overlap is only resolved within a pass -- without this the same
    # physical load is emitted once per phrasing and Cell 10 sees N duplicates.
    dedup=[]
    for c in collected:
        a=float(c["mask"].sum())
        dup=False
        for p in dedup:
            inter=float(_np9.logical_and(c["mask"],p["mask"]).sum())
            if inter/max(1.0,a) >= 0.60:
                dup=True
                if c["score"]>p["score"]: p["score"]=c["score"]
                break
        if not dup: dedup.append(c)
    if len(dedup)!=len(collected):
        print(f"  SAM3: de-duplicated {len(collected)-len(dedup)} repeat hit(s) across "
              f"concept phrasings.")
    collected=dedup

    results=[]; obbs=[]
    exm_full=boxes_to_mask(exclusion_boxes,(H,W)) if exclusion_boxes else None
    for ci,cand in enumerate(collected):
        raw=cand["mask"].astype(_np9.uint8); ss=float(cand["score"])
        if cand["box"] is not None:
            target=_np9.asarray(cand["box"],dtype=_np9.float32)
        else:                                   # no box -> derive from the mask
            ys0,xs0=_np9.where(cand["mask"])
            if len(xs0)==0: continue
            target=_np9.array([xs0.min(),ys0.min(),xs0.max(),ys0.max()],dtype=_np9.float32)
        info={}
        clean=clean_cargo_mask(raw,image_rgb,target,exm_full,info)
        if clean.sum()==0:
            clean=clean_cargo_mask(raw,image_rgb,target,None)
        clean=(clean>0)
        ys9,xs9=_np9.where(clean)
        if len(xs9)==0: continue
        tight=_np9.array([xs9.min(),ys9.min(),xs9.max(),ys9.max()],dtype=_np9.float32)
        yc=float(confs[ci]) if ci<len(confs) else (float(confs[-1]) if confs else 0.5)
        lbl=str(labels[ci]) if ci<len(labels) else "cargo_payload"
        bs=candidate_selection_score_probe(clean,target,lbl,yc,ss,image_rgb.shape)
        obb=None
        try:
            p9=_np9.column_stack([xs9,ys9]).astype(_np9.float32)
            rr=_cv9.minAreaRect(p9); op=_cv9.boxPoints(rr).astype(_np9.float32)
            oa=abs(_cv9.contourArea(op.reshape(-1,1,2)))
            aw,ah=float(tight[2]-tight[0]),float(tight[3]-tight[1])
            ew,eh=float(op[:,0].max()-op[:,0].min()),float(op[:,1].max()-op[:,1].min())
            if oa>0 and oa<OBB_TIGHTNESS_RATIO*max(aw*ah,1.0) and ew<=1.15*max(aw,1.0) and eh<=1.15*max(ah,1.0):
                obb=op
        except Exception: obb=None
        results.append((clean,tight,lbl,yc,ss,"FULL",
                        _np9.asarray(target,dtype=_np9.float32),
                        _np9.asarray(target,dtype=_np9.float32),float(bs)))
        obbs.append(("FULL",obb))
    print(f"  SAM3: kept {len(results)} cargo object(s) from {len(collected)} concept hit(s).")
    return results,obbs

all_masks=[];roi_boxes_by_image=[];roi_timings_by_image=[]
with cargo_stage("CELL 9 - segment",total=len(images),unit="image") as img_stage:
  for image_index, image in enumerate(images):
    img_stage.set_note(f"image {image_index+1}/{len(images)}")
    image_t0=_t9.perf_counter()
    image_rgb=_np9.asarray(image).copy()
    if image_rgb.dtype!=_np9.uint8: image_rgb=_np9.clip(image_rgb,0,255).astype(_np9.uint8)
    H,W=image_rgb.shape[:2]
    rois=get_roi_boxes(image_rgb);roi_boxes_by_image.append(rois)
    image_results=[];image_timings={};image_obbs=[]
    t0d=_t9.perf_counter();regions=detect_cargo_regions(image_rgb)
    cargo_done(f"  detect regions [image {image_index+1}]",t0d,
        objects=len(regions.get("payload_boxes",[])),source=regions.get("source","?"))
    excl=regions.get("exclusion_boxes",[])
    t0p=_t9.perf_counter();prompts=build_per_object_prompts(regions.get("payload_boxes",[]),
        regions.get("envelope_boxes",[]),excl,image_rgb)
    cargo_done(f"  build prompts [image {image_index+1}]",t0p,prompts=len(prompts))
    labels=regions.get("payload_labels",[]) or ["cargo_payload"]
    confs=list(regions.get("payload_confs",[])) or [0.5]

    # ===== SAM 3 concept path (PRIMARY) =====================================
    # Concept grounding does not need per-object ROIs: the whole image is encoded
    # once and every prompt is answered against that embedding. Text prompts also
    # recover loads YOLO MISSED entirely, which is the recall gain.
    if USE_SAM3:
        t0=_t9.perf_counter()
        try:
            s3_res,s3_obbs=segment_image_sam3(image_rgb,
                regions.get("payload_boxes",[]),regions.get("envelope_boxes",[]),
                excl,labels,confs,image_index)
        except Exception as e:
            print(f"  [WARN] SAM3 path failed ({type(e).__name__}: {e}) -> SAM 1 fallback.")
            s3_res,s3_obbs=[],[]
        if s3_res:
            image_results=s3_res; image_obbs=s3_obbs
            image_timings["SAM3"]={"objects":len(image_results),
                "total_seconds":_t9.perf_counter()-t0,
                "source":f"{regions.get('source','?')}/SAM3_CONCEPT","rank":0.0}
            cargo_done(f"  image {image_index+1} total",image_t0,
                objects=len(image_results),source=f"{regions.get('source','?')}/SAM3_CONCEPT")
            all_masks.append(image_results);cargo_obb_by_image.append(image_obbs)
            roi_timings_by_image.append(image_timings)
            img_stage.update();continue
        # SAM 3 found nothing at all -- fall through to SAM 1 below so a single
        # hard image cannot zero out the whole run. Cell 7 only builds the SAM 1
        # predictor when SAM_BACKEND=="sam1", so build it lazily HERE on first need.
        if predictor is None:
            try:
                _tL=_t9.perf_counter()
                from segment_anything import sam_model_registry, SamPredictor
                _ck=MODEL_DIR / "sam_vit_b_01ec64.pth"
                if not _ck.exists():
                    print("  [WARN] SAM3 found nothing and no SAM 1 checkpoint to fall back to.")
                    prompts=[]
                else:
                    sam=sam_model_registry["vit_b"](checkpoint=str(_ck))
                    sam.to(device); sam.eval()
                    predictor=SamPredictor(sam)
                    cargo_done("  lazy-build SAM 1 fallback",_tL,device=device)
            except Exception as _lf:
                print(f"  [WARN] SAM 1 lazy build failed: {type(_lf).__name__}: {_lf}")
                prompts=[]

    # ===== SAM 1 ViT-B path (fallback / explicit SAM_BACKEND="sam1") ==========
    with cargo_stage(f"  SAM objects [image {image_index+1}]",total=len(prompts),
                     unit="obj",leave=False) as obj_stage:
      for pr in prompts:
        obj_stage.set_note(f"obj {pr['index']}")
        t0=_t9.perf_counter()
        rx1,ry1,rx2,ry2=[int(round(float(v))) for v in pr["roi"]]
        rx1,ry1=max(0,rx1),max(0,ry1);rx2,ry2=min(W,rx2),min(H,ry2)
        if rx2<=rx1 or ry2<=ry1: continue
        crop=image_rgb[ry1:ry2,rx1:rx2];ch,cw=crop.shape[:2]
        sc=min(3.0,320.0/max(8.0,min(ch,cw))) if min(ch,cw)<320 else 1.0
        if sc>1.0: crop=_cv9.resize(crop,None,fx=sc,fy=sc,interpolation=_cv9.INTER_CUBIC)
        bx=[(float(pr["box"][0])-rx1)*sc,(float(pr["box"][1])-ry1)*sc,
            (float(pr["box"][2])-rx1)*sc,(float(pr["box"][3])-ry1)*sc]
        allp=_np9.array([[(x-rx1)*sc,(y-ry1)*sc] for x,y in pr["pos"]+pr["neg"]],dtype=_np9.float32)
        alll=_np9.array([1]*len(pr["pos"])+[0]*len(pr["neg"]),dtype=_np9.int32)
        tC=_t9.perf_counter()
        cargo_done(f"    crop/upscale obj{pr['index']}",tC,size=f"{cw}x{ch}",scale=f"{sc:.2f}")
        tE=_t9.perf_counter();predictor.set_image(crop);enc=_t9.perf_counter()-tE
        cargo_done(f"    SAM encode obj{pr['index']}",tE,shape=f"{crop.shape[1]}x{crop.shape[0]}")
        try:
            tD=_t9.perf_counter()
            ms,ss,_=predictor.predict(box=_np9.array(bx,dtype=_np9.float32),
                point_coords=allp,point_labels=alll,multimask_output=True)
            dec=[_t9.perf_counter()-tD]
            cargo_done(f"    SAM predict obj{pr['index']}",tD,masks=len(ms))
        except Exception as e:
            print(f"  [WARN] SAM failed obj{pr['index']}: {type(e).__name__}: {e}")
            image_timings[f"OBJ{pr['index']}"]={"objects":len(image_results),
                "total_seconds":_t9.perf_counter()-t0,"source":"SAM_ERROR"};continue
        obj_stage.update()
        exm=None
        if excl:
            exm=boxes_to_mask([[(float(b[0])-rx1)*sc,(float(b[1])-ry1)*sc,
                (float(b[2])-rx1)*sc,(float(b[3])-ry1)*sc] for b in excl],crop.shape)
        ci=pr["index"];yc=float(confs[min(ci,len(confs)-1)])
        tR=_t9.perf_counter()
        bi,bs,rows=rank_sam_candidates(ms,ss,_np9.array(bx,dtype=_np9.float32),yc,crop,exm,pr["y_start"])
        cargo_done(f"    rank candidates obj{pr['index']}",tR,best=f"{bi}")
        if bi is None: continue
        winner=(ms[bi]>0).astype(_np9.uint8)
        _info={}
        tM=_t9.perf_counter()
        clean=clean_cargo_mask(winner,crop,_np9.array(bx,dtype=_np9.float32),exm,_info)
        # back off the exclusion subtraction when the load band is mostly white cargo:
        # if the ROI column is >55% bright (>=200), exclusions subtracted too much —
        # restore the winner inside the prompt box (dynamic brightness test, no fixed color).
        # Skipped when the winner was a non-discriminative blob (already clamped above).
        grayc=_cv9.cvtColor(crop,_cv9.COLOR_RGB2GRAY)
        bright_frac=float((grayc>=200).mean())
        if (exm is not None and not _info.get("non_discriminative") and bright_frac>0.55
                and clean.sum()<0.5*float(winner.sum())):
            clean=((winner>0).astype(_np9.uint8))
            clean[scene_background_mask(crop)]=0
            nl2,lb2,st2,_=_cv9.connectedComponentsWithStats(clean)
            if nl2>1:
                big2=1+int(_np9.argmax(st2[1:,_cv9.CC_STAT_AREA]))
                clean=(lb2==big2).astype(_np9.uint8)
        if clean.sum()==0: clean=clean_cargo_mask(winner,crop,_np9.array(bx,dtype=_np9.float32),None)
        cargo_done(f"    clean mask obj{pr['index']}",tM,px=int(clean.sum()))
        if sc!=1.0: clean=_cv9.resize(clean,(cw,ch),interpolation=_cv9.INTER_NEAREST)
        clean=(clean>0)
        full=_np9.zeros((H,W),dtype=bool)
        hh2,ww2=min(H-ry1,clean.shape[0]),min(W-rx1,clean.shape[1])
        full[ry1:ry1+hh2,rx1:rx1+ww2]=clean[:hh2,:ww2]
        ys9,xs9=_np9.where(full)
        if len(xs9)==0: continue
        tight=_np9.array([xs9.min(),ys9.min(),xs9.max(),ys9.max()],dtype=_np9.float32)
        sb=_np9.array(bx,dtype=_np9.float32);sb[[0,2]]/=sc;sb[[1,3]]/=sc
        sb[[0,2]]+=rx1;sb[[1,3]]+=ry1
        yf=_np9.asarray(pr["yolo_box"],dtype=_np9.float32)
        obb=None
        try:
            p9=_np9.column_stack([xs9,ys9]).astype(_np9.float32)
            rr=_cv9.minAreaRect(p9);op=_cv9.boxPoints(rr).astype(_np9.float32)
            oa=abs(_cv9.contourArea(op.reshape(-1,1,2)))
            aw,ah=float(tight[2]-tight[0]),float(tight[3]-tight[1])
            ew,eh=float(op[:,0].max()-op[:,0].min()),float(op[:,1].max()-op[:,1].min())
            if oa>0 and oa<OBB_TIGHTNESS_RATIO*max(aw*ah,1.0) and ew<=1.15*max(aw,1.0) and eh<=1.15*max(ah,1.0):
                obb=op
        except Exception: obb=None
        image_results.append((full,tight,str(labels[min(ci,len(labels)-1)]),
            float(yc),float(ss[bi]),"FULL",_np9.asarray(yf,dtype=_np9.float32),
            _np9.asarray(sb,dtype=_np9.float32),float(bs)))
        image_obbs.append(("FULL",obb))
        image_timings[f"OBJ{ci}"]={"objects":len(image_results),"decode_times":dec,
            "encode_seconds":enc,"total_seconds":_t9.perf_counter()-t0,
            "source":f"{regions.get('source','?')}/{pr['source']}","rank":float(bs)}
        cargo_done(f"    OBJECT {pr['index']} TOTAL [image {image_index+1}]",t0,
            kept=len(image_results),rank=f"{bs:.2f}")
    all_masks.append(image_results);cargo_obb_by_image.append(image_obbs)
    roi_timings_by_image.append(image_timings)
    cargo_done(f"  image {image_index+1} total",image_t0,
        objects=len(image_results),source=regions.get("source","?"))
    img_stage.update()
print("Cell 9 ready: per-object SAM masks + tight boxes on EVERY image | segmenter =",
      "SAM 3 concept prompts" if USE_SAM3 else "SAM 1 ViT-B")





In [ ]:
# ============================================================
# NEW CELL 9b — HIGH-PRECISION SELF-TRAINING ENGINE (YOLO11x @ 1024px)
# ============================================================
import os
import random
import shutil
import cv2
import numpy as np
from pathlib import Path
from ultralytics import YOLO

dataset_root = Path("./self_train_cargo_dataset")
train_img_dir = dataset_root / "images" / "train"
val_img_dir = dataset_root / "images" / "val"
train_lbl_dir = dataset_root / "labels" / "train"
val_lbl_dir = dataset_root / "labels" / "val"

train_img_dir.mkdir(parents=True, exist_ok=True)
val_img_dir.mkdir(parents=True, exist_ok=True)
train_lbl_dir.mkdir(parents=True, exist_ok=True)
val_lbl_dir.mkdir(parents=True, exist_ok=True)

# Self-training is OFF by default. It trains a 1-class detector on the tiny
# ./self_train_cargo_dataset (built from the 1-3 selected images) which CELL 7
# never loads -- CELL 7 loads CARGO_YOLO_WEIGHTS, the zip-trained 72/18 OBB model.
# An interrupted run also reports mAP50 = 0, which says nothing about the real
# detector. Set CARGO_SELF_TRAIN=1 to run it as an experiment.
SELF_TRAIN_ENABLED = os.environ.get("CARGO_SELF_TRAIN", "0").strip().lower() not in (
    "0", "false", "no", "off")

print("Step 1: Compiling precision SAM mask tracks into train/val dataset splits...")

all_indices = list(range(len(images)))
random.seed(42)
random.shuffle(all_indices)

split_idx = max(1, int(0.8 * len(all_indices)))
train_indices = set(all_indices[:split_idx])

for idx, (image, image_results) in enumerate(zip(images, all_masks)):
    image_rgb = np.asarray(image)
    h, w, _ = image_rgb.shape
    
    is_train = idx in train_indices or len(all_indices) == 1
    img_target = train_img_dir if is_train else val_img_dir
    lbl_target = train_lbl_dir if is_train else val_lbl_dir
    
    cv2.imwrite(str(img_target / f"cargo_{idx:03d}.jpg"), cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR))
    label_lines = []
    
    for mask_data in image_results:
        local_mask = mask_data[0]  
        y_indices, x_indices = np.where(local_mask)
        
        if len(x_indices) == 0 or len(y_indices) == 0:
            continue
            
        x1, y1 = np.min(x_indices), np.min(y_indices)
        x2, y2 = np.max(x_indices), np.max(y_indices)
        
        bw, bh = (x2 - x1) / w, (y2 - y1) / h
        cx, cy = (x1 + (x2 - x1) / 2) / w, (y1 + (y2 - y1) / 2) / h
        
        # Class 2: suspended_cargo_in_net
        label_lines.append(f"2 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
        
    with open(str(lbl_target / f"cargo_{idx:03d}.txt"), "w") as f:
        f.write("\n".join(label_lines))

# If val is empty due to small image count, duplicate at least 1 image to val to prevent DataLoader errors
val_images = list(val_img_dir.glob("*"))
if len(val_images) == 0 and len(list(train_img_dir.glob("*"))) > 0:
    first_train_img = list(train_img_dir.glob("*"))[0]
    first_train_lbl = train_lbl_dir / f"{first_train_img.stem}.txt"
    shutil.copy2(first_train_img, val_img_dir / first_train_img.name)
    if first_train_lbl.exists():
        shutil.copy2(first_train_lbl, val_lbl_dir / first_train_lbl.name)

print("Step 1 Complete: Dataset train/val split created successfully.")

if SELF_TRAIN_ENABLED:
    print("=" * 74)
    print("  STARTING SELF-TRAINING (yolo11x) on ./self_train_cargo_dataset")
    print("=" * 74)
else:
    print("=" * 74)
    print("[cell 9b] self-training DISABLED (CARGO_SELF_TRAIN=0) - train loop skipped.")
    print("          The pipeline detector stays CARGO_YOLO_WEIGHTS: the")
    print("          zip-trained OBB model on the 72/18 scene-grouped split.")
    print("          Set CARGO_SELF_TRAIN=1 to run this experiment. Its weights are")
    print("          not promoted either way, and a partial run reports mAP50=0,")
    print("          which says nothing about the real detector.")
    print("=" * 74)

custom_yolo = YOLO("yolo11x.pt") if SELF_TRAIN_ENABLED else None

training_metrics = (custom_yolo.train(
    data="cargo_auto_data.yaml",
    epochs=30,  # CPU-friendly: was 100; patience=10 below usually stops even earlier          
    imgsz=640,  # drop to 640 for ~2.6x faster epochs if the PC still stutters          
    batch=2,            
    device=device,
    workers=0,      # in-process dataloader: no worker procs competing for the CPU
    patience=10,    # early-stop when val mAP stalls -> saves the remaining epochs
    
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,          
    mosaic=1.0,         
    mixup=0.15,
    copy_paste=0.3,
    close_mosaic=5,    
    
    box=8.5,            
    cls=0.5,
    dfl=2.0,            
    
    lr0=0.001,          
    lrf=0.01,           
    warmup_epochs=3,    
    plots=True,
    verbose=True
) if SELF_TRAIN_ENABLED else None)

# Keep the zip-aware taxonomy intact: reload the RESOLVED cargo detector.
_cargo_weights = globals().get("CARGO_YOLO_WEIGHTS", "yolo11x.pt")
yolo_model = YOLO(str(_cargo_weights))
print(f"Cell 9b: cargo detector reloaded from {_cargo_weights}")

print("\n[SUCCESS]: High-precision custom weight optimization epochs complete.")


In [ ]:
# ============================================================
# CELL 10 — CANDIDATE SCORING, CARGO-ONLY VISUALIZATION & DASHBOARD
# ============================================================
# WHAT: validate EVERY cargo object; RED=tight box, ORANGE=mask; debug YOLO/SAM boxes.
import time as _t10
import cv2 as _cv10
import matplotlib.pyplot as _plt10
import numpy as _np10
ROI_NAMES=("FULL","LOAD","CARGO")
selected_masks=[];cargo_geometry=[];refinement_results=[];selected_boxes=[]
MASK_COLOR=_np10.asarray([255,165,0],dtype=_np10.float32)  # ORANGE mask
BOX_COLOR_RGB=(255,0,0)   # RED box
OBB_COLOR_RGB=(255,170,0); BOX_THICKNESS=4
DEBUG_VIZ=False  # True -> also draw YOLO raw (dashed) + SAM raw outlines
MIN_ACCEPT_SCORE=6.0; MAX_ACCEPT_BOX_FRACTION=0.60; MAX_ACCEPT_BORDER_LEAK=0.12
def _mask_border_leak(m):
    if m is None or m.size==0: return 0.0
    r=_np10.zeros(m.shape,dtype=bool);r[:3,:]=True;r[-3:,:]=True;r[:,:3]=True;r[:,-3:]=True
    return float((m&r).sum())/max(1.0,float(r.sum()))
def mask_to_full_image(m, roi_name, roi_boxes, shape):
    H,W=shape[:2];f=_np10.zeros((H,W),dtype=bool)
    if roi_name=="FULL":
        th,tw=min(H,m.shape[0]),min(W,m.shape[1]);f[:th,:tw]=m[:th,:tw];return f
    rb=roi_boxes.get(roi_name)
    if rb is None: return f
    x1,y1,x2,y2=rb;th=min(y2-y1,m.shape[0]);tw=min(x2-x1,m.shape[1])
    f[y1:y1+th,x1:x1+tw]=m[:th,:tw];return f
def candidate_selection_score(item, shape):
    mask,box,label,conf,sam,roi=item[:6]
    H,W=shape[:2];A=float(H*W);ma=float(mask.sum())
    if ma<=0: return -500.0
    x1,y1,x2,y2=[float(v) for v in box];bw,bh=max(1.0,x2-x1),max(1.0,y2-y1)
    ba=bw*bh
    ys,xs=_np10.where(mask)
    mba=float(max(1,xs.max()-xs.min()+1)*max(1,ys.max()-ys.min()+1)) if len(ys) else ba
    fill=min(1.0,ma/max(1.0,mba));ar=min(1.0,ma/max(1.0,A))
    area=max(0.0,1.0-abs(ar-0.25)/0.75)
    if ar<0.03: area=0.0
    over=8.0*max(0.0,ar-0.40);under=3.0*max(0.0,(0.03-ar)/0.03)
    floor=10.0 if y2>=(H-5) else 0.0;top=2.0 if y1<=2 else 0.0
    bord=6.0*max(0.0,_mask_border_leak(mask)-0.05)
    fb=2.0 if label=="cargo_payload" else 0.0
    # crane-slat guard: a big mask that is mostly EMPTY (low fill) and pinned to the
    # TOP edge is rigging/sky, not cargo. Relative terms only (top rows + fill).
    sky=6.0 if (y1<=3 and fill<0.45) else 0.0
    return float(8.0*float(sam)+6.0*fill+4.0*float(conf)+2.0*area-floor-top-over-under-bord-fb-sky)
with cargo_stage("CELL 10 - score & visualize",total=len(images),unit="image") as viz_stage:
 for image_index, image in enumerate(images):
    viz_stage.set_note(f"image {image_index+1}/{len(images)}")
    image_t0=_t10.perf_counter()
    image_rgb=_np10.asarray(image).copy()
    if image_rgb.dtype!=_np10.uint8: image_rgb=_np10.clip(image_rgb,0,255).astype(_np10.uint8)
    H,W=image_rgb.shape[:2]
    objects=all_masks[image_index]
    roi_boxes=roi_boxes_by_image[image_index]
    obbs=dict(cargo_obb_by_image[image_index]) if image_index<len(cargo_obb_by_image) else {}
    vis=image_rgb.copy()
    selected_masks.append(None);cargo_geometry.append(None)
    refinement_results.append(objects);selected_boxes.append(None)
    if not objects:
        print(f"Image {image_index+1}: no candidate masks available.")
        op=OUTPUT_DIR/f"cargo_yolo_sam_image_{image_index+1}.png"
        _cv10.imwrite(str(op),_cv10.cvtColor(vis,_cv10.COLOR_RGB2BGR))
        vd=PROJECT_DIR/"cargo_output_visualizations";vd.mkdir(exist_ok=True)
        _cv10.imwrite(str(vd/f"cargo_yolo_sam_image_{image_index+1}.png"),_cv10.cvtColor(vis,_cv10.COLOR_RGB2BGR))
        _plt10.figure(figsize=(12,9));_plt10.imshow(vis)
        _plt10.title(f"No cargo detected — Image {image_index+1}");_plt10.axis("off")
        _plt10.tight_layout();_plt10.show()
        cargo_done(f"  score/draw [image {image_index+1}]",image_t0,result="no-candidates")
        viz_stage.update();continue
    t0s=_t10.perf_counter()
    ranked=sorted(objects,key=lambda it:candidate_selection_score(it,image_rgb.shape),reverse=True)
    kept=[]
    for it in ranked:
        sc=candidate_selection_score(it,image_rgb.shape)
        m,b=it[0],[float(v) for v in it[1]]
        frac=((b[2]-b[0])*(b[3]-b[1]))/max(1.0,float(H*W));leak=_mask_border_leak(m)
        if sc<MIN_ACCEPT_SCORE or frac>MAX_ACCEPT_BOX_FRACTION or leak>MAX_ACCEPT_BORDER_LEAK:
            continue
        kept.append((it,sc,frac,leak))
    if not kept:
        best=ranked[0];sc=candidate_selection_score(best,image_rgb.shape)
        b=[float(v) for v in best[1]]
        frac=((b[2]-b[0])*(b[3]-b[1]))/max(1.0,float(H*W));leak=_mask_border_leak(best[0])
        print(f"Image {image_index+1}: NO CONFIDENT CARGO -- best candidate score {sc:.1f} "
              f"(min {MIN_ACCEPT_SCORE}), box {frac:.0%} of frame (max {MAX_ACCEPT_BOX_FRACTION:.0%}), "
              f"border leak {leak:.0%} (max {MAX_ACCEPT_BORDER_LEAK:.0%}). No box drawn.")
        op=OUTPUT_DIR/f"cargo_yolo_sam_image_{image_index+1}.png"
        _cv10.imwrite(str(op),_cv10.cvtColor(vis,_cv10.COLOR_RGB2BGR))
        vd=PROJECT_DIR/"cargo_output_visualizations";vd.mkdir(exist_ok=True)
        _cv10.imwrite(str(vd/f"cargo_yolo_sam_image_{image_index+1}.png"),_cv10.cvtColor(vis,_cv10.COLOR_RGB2BGR))
        _plt10.figure(figsize=(12,9));_plt10.imshow(vis)
        _plt10.title(f"No confident cargo detected -- Image {image_index+1}");_plt10.axis("off")
        _plt10.tight_layout();_plt10.show()
        cargo_done(f"  score/draw [image {image_index+1}]",image_t0,result="rejected",score=f"{sc:.1f}")
        viz_stage.update();continue
    cargo_done(f"  rank + acceptance gate [image {image_index+1}]",t0s,
        candidates=len(ranked),kept=len(kept))
    print(f"Image {image_index+1}: accepted '{ranked[0][2]}' "
          f"(score {kept[0][1]:.1f}, box {kept[0][2]:.0%} of frame, border leak {kept[0][3]:.0%}) "
          f"x{len(kept)} object(s).")
    for oi,((m,b,label,conf,sam,roi,*rest),sc,frac,leak) in enumerate(kept, start=1):
        yb=rest[0] if len(rest)>=1 else _np10.asarray(b)
        sb=rest[1] if len(rest)>=2 else _np10.asarray(b)
        fm=mask_to_full_image(m,roi,roi_boxes,image_rgb.shape)
        area=int(fm.sum())
        ys,xs=_np10.where(fm)
        if len(xs)==0: continue
        tight=[int(xs.min()),int(ys.min()),int(xs.max()),int(ys.max())]
        cnts,_=_cv10.findContours(fm.astype(_np10.uint8),_cv10.RETR_EXTERNAL,_cv10.CHAIN_APPROX_SIMPLE)
        cnt=max(cnts,key=_cv10.contourArea) if cnts else None
        print(f"  Cargo #{oi}: class={label} yolo={conf:.2f} sam={sam:.2f} mask_area={area}px "
              f"yolo_box={[round(float(v)) for v in yb]} sam_box={[round(float(v)) for v in sb]} tight={tight}")
        vis[fm]=(0.55*vis[fm].astype(_np10.float32)+0.45*MASK_COLOR).astype(_np10.uint8)
        _cv10.rectangle(vis,(tight[0],tight[1]),(tight[2],tight[3]),BOX_COLOR_RGB,max(2,W//320))
        _cv10.putText(vis,f"Cargo #{oi} {label} {conf:.2f}",(max(0,tight[0]),max(24,tight[1]-8)),
            _cv10.FONT_HERSHEY_SIMPLEX,0.55,BOX_COLOR_RGB,2,_cv10.LINE_AA)
        if oi==1:
            selected_masks[-1]=fm;selected_boxes[-1]=_np10.array(tight,dtype=_np10.float32)
            cargo_geometry[-1]={"mask_area":area,"tight_box":tight,"contour":cnt,
                "dims_px":[tight[2]-tight[0],tight[3]-tight[1]]}
    op=OUTPUT_DIR/f"cargo_yolo_sam_image_{image_index+1}.png"
    _cv10.imwrite(str(op),_cv10.cvtColor(vis,_cv10.COLOR_RGB2BGR))
    vd=PROJECT_DIR/"cargo_output_visualizations";vd.mkdir(exist_ok=True)
    _cv10.imwrite(str(vd/f"cargo_yolo_sam_image_{image_index+1}.png"),_cv10.cvtColor(vis,_cv10.COLOR_RGB2BGR))
    _plt10.figure(figsize=(12,9));_plt10.imshow(vis)
    _plt10.title(f"Selected Tight Cargo Objects ({len(kept)}) -- Image {image_index+1}")
    _plt10.axis("off");_plt10.tight_layout();_plt10.show()
    cargo_done(f"  score/draw [image {image_index+1}]",image_t0,
        result="accepted",objects=len(kept))
    viz_stage.update()
print("\n"+"="*62+"\nPERFORMANCE SUMMARY DASHBOARD\n"+"="*62)
print("| ROI    | Objects | Source                     | Total Processing Time |")
print("|"+"-"*60+"|")
for tm in roi_timings_by_image:
    for n,d in tm.items():
        print(f"| {n:<6} | {d['objects']:>7} | {str(d.get('source','-'))[:26]:<26} | {d['total_seconds']:>20.4f}s |")
print("="*62)
for i,b in enumerate(selected_boxes):
    if b is None: continue
    a,b2,c,d=[float(v) for v in b]
    print(f"Image {i+1}: cargo box = ({a:.0f}, {b2:.0f}) -> ({c:.0f}, {d:.0f})  [{c-a:.0f} x {d-b2:.0f} px]")

# --- telemetry: per-cell + per-step timings collected by the bootstrap cell --
# This is a PARTIAL profile: cells 11-28 (calibration, depth, point cloud,
# volume, weight) have not run yet. Nothing needs saving here -- the CELL 2
# post_run_cell hook already wrote outputs/timings.* after this very cell.
cargo_timing_report(partial=True)





In [ ]:
# ============================================================
# CELL 11 — VISUALIZE SELECTED CARGO MASK
# ============================================================

for idx, (image, mask) in enumerate(
    zip(images, selected_masks)
):

    if mask is None:
        continue

    overlay = image.copy()

    overlay[mask] = (
        0.5 * overlay[mask]
        + 0.5 * np.array([0, 255, 0])
    )

    plt.figure(figsize=(10, 7))

    plt.imshow(overlay.astype(np.uint8))

    plt.title(
        f"Cargo Segmentation — Image {idx + 1}"
    )

    plt.axis("off")
    plt.show()


In [ ]:
# ============================================================
# CELL 12 — EXTRACT CARGO GEOMETRY
# ============================================================

cargo_geometry = []

for idx, mask in enumerate(selected_masks):

    if mask is None:
        cargo_geometry.append(None)
        continue

    ys, xs = np.where(mask)

    x_min = int(xs.min())
    x_max = int(xs.max())

    y_min = int(ys.min())
    y_max = int(ys.max())

    width_px = x_max - x_min
    height_px = y_max - y_min

    area_px = int(np.sum(mask))

    geometry = {
        "x_min": x_min,
        "x_max": x_max,
        "y_min": y_min,
        "y_max": y_max,
        "width_px": width_px,
        "height_px": height_px,
        "area_px": area_px
    }

    cargo_geometry.append(geometry)

    print(f"\nImage {idx + 1}")
    print(json.dumps(geometry, indent=2))


In [ ]:
# ============================================================
# CELL 13 — REFERENCE SCALE
# ============================================================

print("""
REFERENCE SCALE CONFIGURATION

Enter a known physical dimension visible in the image.

Example:
If the crane net width is known to be 2.0 meters,
we can use that as our reference.
""")

REFERENCE_WIDTH_METERS = 2.0

# For initial PoC demonstration:
# user can manually select two points representing
# the known reference distance.

print(
    f"Reference width = "
    f"{REFERENCE_WIDTH_METERS} meters"
)


In [ ]:
# ============================================================
# CELL 14 — FULLY AUTOMATIC PIXEL-TO-METER CALIBRATION
#            NO USER INPUT
# ============================================================

import numpy as np
import cv2


# ============================================================
# AUTOMATIC REFERENCE DETECTION
#
# Uses SAM masks to identify the crane/net reference structure.
# The reference width is taken from the widest suitable
# non-background SAM region.
# ============================================================

def find_reference_width_pixels(
    image,
    masks
):

    image_h, image_w = image.shape[:2]
    image_area = image_h * image_w

    candidates = []

    for mask_data in masks:

        # Accept BOTH the pipeline's 6-tuple (mask, box, label, conf, sam, roi)
        # and legacy SAM dict masks ({"segmentation": ..., "area": ...}).
        if isinstance(mask_data, dict):
            mask = mask_data["segmentation"]
            area = int(mask_data["area"])
        else:
            mask = mask_data[0]
            area = int(np.sum(mask))

        area_ratio = area / image_area

        ys, xs = np.where(mask)

        if len(xs) == 0:
            continue

        x_min = int(xs.min())
        x_max = int(xs.max())

        y_min = int(ys.min())
        y_max = int(ys.max())

        width = x_max - x_min
        height = y_max - y_min

        if width <= 0 or height <= 0:
            continue

        width_ratio = width / image_w
        height_ratio = height / image_h

        # ----------------------------------------------------
        # We are looking for a large structural object
        # such as the crane net surrounding the cargo.
        #
        # Unlike the cargo detector, the reference structure
        # is allowed to touch image boundaries.
        # ----------------------------------------------------

        if width_ratio < 0.35:
            continue

        if height_ratio < 0.25:
            continue

        if area_ratio < 0.05:
            continue

        if area_ratio > 0.90:
            continue

        # ----------------------------------------------------
        # Prefer wide structures
        # ----------------------------------------------------

        width_score = width_ratio

        # ----------------------------------------------------
        # Prefer reasonably tall structures
        # ----------------------------------------------------

        height_score = min(
            height_ratio / 0.60,
            1.0
        )

        # ----------------------------------------------------
        # Prefer masks with moderate fill ratio.
        # Very dense image-wide masks are likely background.
        # ----------------------------------------------------

        bbox_area = width * height

        fill_ratio = (
            area / bbox_area
            if bbox_area > 0
            else 0
        )

        fill_score = min(
            fill_ratio * 2.0,
            1.0
        )

        # ----------------------------------------------------
        # SAM quality
        # ----------------------------------------------------

        if isinstance(mask_data, dict):
            stability_score = float(mask_data.get("stability_score", 0.0))
            predicted_iou = float(mask_data.get("predicted_iou", 0.0))
        else:
            # Pipeline 6-tuple has no SAM metadata; index 4 is its quality score.
            _quality = float(mask_data[4])
            stability_score = _quality
            predicted_iou = _quality

        # ----------------------------------------------------
        # Final reference score
        # ----------------------------------------------------

        score = (

            width_score * 0.45

            + height_score * 0.20

            + fill_score * 0.10

            + stability_score * 0.15

            + predicted_iou * 0.10
        )

        candidates.append({

            "mask": mask,

            "width": width,

            "height": height,

            "area": area,

            "area_ratio": area_ratio,

            "width_ratio": width_ratio,

            "height_ratio": height_ratio,

            "fill_ratio": fill_ratio,

            "score": score
        })


    # ========================================================
    # NO REFERENCE FOUND
    # ========================================================

    if len(candidates) == 0:

        return None


    # ========================================================
    # SELECT BEST REFERENCE STRUCTURE
    # ========================================================

    candidates = sorted(
        candidates,
        key=lambda x: x["score"],
        reverse=True
    )

    return candidates[0]


# ============================================================
# AUTOMATIC CALIBRATION
# ============================================================

_t14 = cargo_step("14. auto pixel-to-meter calibration")
reference_candidate = find_reference_width_pixels(
    images[0],
    all_masks[0]
)
cargo_done("14. auto pixel-to-meter calibration", _t14,
    found=reference_candidate is not None)


# ============================================================
# FALLBACK
# ============================================================

if reference_candidate is None:

    # Conservative fallback:
    # use a fixed proportion of the image width.
    #
    # This keeps the pipeline fully automatic, but the
    # resulting metric scale is only an approximation.

    reference_pixel_width = (
        images[0].shape[1] * 0.65
    )

    print(
        "Automatic reference structure not detected."
    )

    print(
        "Using automatic fallback reference width."
    )

else:

    reference_pixel_width = (
        reference_candidate["width"]
    )

    print(
        "Automatic reference structure detected."
    )

    print(
        "Reference pixel width:",
        reference_pixel_width
    )

    print(
        "Reference detection score:",
        round(
            reference_candidate["score"],
            3
        )
    )


# ============================================================
# PIXEL → METER CONVERSION
# ============================================================

meters_per_pixel = (
    REFERENCE_WIDTH_METERS
    / reference_pixel_width
)

pixels_per_meter = (
    reference_pixel_width
    / REFERENCE_WIDTH_METERS
)


# ============================================================
# RESULTS
# ============================================================

print()
print("=" * 55)
print("AUTOMATIC SCALE CALIBRATION")
print("=" * 55)

print(
    f"Known reference width : "
    f"{REFERENCE_WIDTH_METERS:.3f} m"
)

print(
    f"Reference pixel width : "
    f"{reference_pixel_width:.2f} px"
)

print(
    f"Meters per pixel      : "
    f"{meters_per_pixel:.8f} m/px"
)

print(
    f"Pixels per meter      : "
    f"{pixels_per_meter:.2f} px/m"
)

print("=" * 55)


In [ ]:
# ============================================================
# CELL 15 — ESTIMATE REAL-WORLD 2D DIMENSIONS
# ============================================================

if cargo_geometry[0] is not None:

    geometry = cargo_geometry[0]

    width_m = (
        geometry["width_px"]
        * meters_per_pixel
    )

    height_m = (
        geometry["height_px"]
        * meters_per_pixel
    )

    print("Estimated cargo width :", width_m, "m")
    print("Estimated cargo height:", height_m, "m")


In [ ]:
# ============================================================
# CELL 16 — INSTALL DEPTH ESTIMATION DEPENDENCIES
# ============================================================

%pip install -q transformers

print("Depth estimation dependencies installed.")


In [ ]:
# ============================================================
# CELL 17 — LOAD DEPTH MODEL
# ============================================================

import torch

from transformers import (
    pipeline
)

_t17 = cargo_step("17. load Depth-Anything model")
depth_estimator = pipeline(
    task="depth-estimation",
    model="LiheYoung/depth-anything-small-hf"
)
cargo_done("17. load Depth-Anything model", _t17, model="depth-anything-small-hf")

print("Depth model loaded.")


In [ ]:
# ============================================================
# CELL 18 — GENERATE DEPTH MAP
# ============================================================

depth_maps = []

with cargo_stage("18. depth maps",total=len(images),unit="image") as depth_stage:
 for idx, image in enumerate(images):
    depth_stage.set_note(f"image {idx+1}/{len(images)}")
    _t18 = cargo_step(f"  depth infer [image {idx+1}]")

    print(f"Generating depth for image {idx + 1}")

    pil_image = Image.fromarray(image)

    result = depth_estimator(
        pil_image
    )

    depth = np.array(
        result["depth"]
    )

    depth_maps.append(depth)
    cargo_done(f"  depth infer [image {idx+1}]", _t18, shape=f"{depth.shape[1]}x{depth.shape[0]}")

    plt.figure(figsize=(10, 7))

    plt.imshow(
        depth,
        cmap="plasma"
    )

    plt.colorbar(
        label="Relative Depth"
    )

    plt.title(
        f"Depth Map — Image {idx + 1}"
    )

    plt.axis("off")

    plt.show()
    depth_stage.update()


In [ ]:
# ============================================================
# CELL 19 — EXTRACT CARGO DEPTH
# ============================================================

cargo_depth_maps = []

for idx, (depth, mask) in enumerate(
    zip(depth_maps, selected_masks)
):

    if mask is None:
        cargo_depth_maps.append(None)
        continue

    # Resize mask if required
    if mask.shape != depth.shape:

        mask_resized = cv2.resize(
            mask.astype(np.uint8),
            (
                depth.shape[1],
                depth.shape[0]
            ),
            interpolation=cv2.INTER_NEAREST
        ).astype(bool)

    else:

        mask_resized = mask

    cargo_depth = np.where(
        mask_resized,
        depth,
        0
    )

    cargo_depth_maps.append(
        cargo_depth
    )

    plt.figure(figsize=(10, 7))

    plt.imshow(
        cargo_depth,
        cmap="plasma"
    )

    plt.title(
        f"Cargo Depth — Image {idx + 1}"
    )

    plt.axis("off")

    plt.show()


In [ ]:
# ============================================================
# CELL 20 — CREATE INITIAL 3D POINT CLOUD
# ============================================================

import open3d as o3d

def depth_to_point_cloud(
    image,
    depth,
    mask=None
):

    h, w = depth.shape

    # Simple normalized camera assumptions
    fx = w
    fy = w

    cx = w / 2
    cy = h / 2

    yy, xx = np.mgrid[
        0:h,
        0:w
    ]

    z = depth.astype(np.float32)

    # Normalize depth
    z = (
        z - z.min()
    ) / (
        z.max() - z.min() + 1e-8
    )

    z = z + 1e-6

    if mask is not None:

        valid = mask

    else:

        valid = np.ones(
            depth.shape,
            dtype=bool
        )

    x = (
        (xx - cx)
        * z
        / fx
    )

    y = (
        (yy - cy)
        * z
        / fy
    )

    points = np.stack(
        [
            x[valid],
            y[valid],
            z[valid]
        ],
        axis=1
    )

    colors = image[
        valid
    ] / 255.0

    pcd = o3d.geometry.PointCloud()

    pcd.points = o3d.utility.Vector3dVector(
        points
    )

    pcd.colors = o3d.utility.Vector3dVector(
        colors
    )

    return pcd


point_clouds = []

with cargo_stage("20. 3D point clouds",total=len(images),unit="image") as pcd_stage:
 for idx, (image, depth, mask) in enumerate(
    zip(
        images,
        depth_maps,
        selected_masks
    )
 ):
    pcd_stage.set_note(f"image {idx+1}/{len(images)}")
    _t20 = cargo_step(f"  build point cloud [image {idx+1}]")

    if mask is None:
        point_clouds.append(None)
        cargo_done(f"  build point cloud [image {idx+1}]", _t20, status="skipped",
            reason="no mask")
        pcd_stage.update(); continue

    if mask.shape != depth.shape:

        mask = cv2.resize(
            mask.astype(np.uint8),
            (
                depth.shape[1],
                depth.shape[0]
            ),
            interpolation=cv2.INTER_NEAREST
        ).astype(bool)

    pcd = depth_to_point_cloud(
        image,
        depth,
        mask
    )

    point_clouds.append(pcd)
    cargo_done(f"  build point cloud [image {idx+1}]", _t20, points=len(pcd.points))

    print(
        f"Image {idx + 1}: "
        f"{len(pcd.points)} points"
    )
    pcd_stage.update()


In [ ]:
# ============================================================
# CELL 21 — VISUALIZE 3D POINT CLOUD IN VS CODE
# ============================================================

import numpy as np
import plotly.graph_objects as go

for idx, pcd in enumerate(point_clouds):

    if pcd is None:
        print(f"Skipping image {idx + 1} — point cloud is None.")
        continue

    points = np.asarray(pcd.points)

    if points.shape[0] == 0:
        print(f"Skipping image {idx + 1} — point cloud contains no points.")
        continue

    colors = np.asarray(pcd.colors)

    marker = dict(size=2, opacity=0.8)

    if colors.shape == points.shape:
        marker["color"] = [
            f"rgb({int(r * 255)},{int(g * 255)},{int(b * 255)})"
            for r, g, b in np.clip(colors, 0, 1)
        ]

    fig = go.Figure(
        data=[go.Scatter3d(
            x=points[:, 0],
            y=points[:, 1],
            z=points[:, 2],
            mode="markers",
            marker=marker,
            name=f"Cargo {idx + 1}"
        )]
    )

    fig.update_layout(
        title=f"3D Cargo Point Cloud — Image {idx + 1}",
        scene=dict(
            xaxis_title="X",
            yaxis_title="Y",
            zaxis_title="Z",
            aspectmode="data"
        ),
        width=1000,
        height=750,
        margin=dict(l=0, r=0, b=0, t=50)
    )

    fig.show()


In [ ]:
# ============================================================
# CELL 22 — INITIAL 3D VOLUME ESTIMATION
# ============================================================

def estimate_bounding_box_volume(pcd):

    bbox = pcd.get_axis_aligned_bounding_box()

    extent = bbox.get_extent()

    volume = (
        extent[0]
        * extent[1]
        * extent[2]
    )

    return volume, extent


volumes = []

with cargo_stage("22. 3D volume",total=len(point_clouds),unit="image") as vol_stage:
 for idx, pcd in enumerate(point_clouds):
    vol_stage.set_note(f"image {idx+1}/{len(point_clouds)}")
    _t22 = cargo_step(f"  volume [image {idx+1}]")

    if pcd is None:
        volumes.append(None)
        cargo_done(f"  volume [image {idx+1}]", _t22, status="skipped")
        vol_stage.update(); continue

    volume, extent = (
        estimate_bounding_box_volume(
            pcd
        )
    )

    volumes.append(volume)
    cargo_done(f"  volume [image {idx+1}]", _t22, volume=f"{volume:.4f}")

    print(f"\nImage {idx + 1}")

    print("X:", extent[0])
    print("Y:", extent[1])
    print("Z:", extent[2])

    print(
        "Relative volume:",
        volume
    )
    vol_stage.update()


In [ ]:
# ============================================================
# CELL 23 — CARGO CLASSIFICATION
# ============================================================

print("""
Available PoC cargo categories:

1. Scrap Steel
2. Thermal Coal
3. Construction Gravel
4. Agricultural Grain
""")

import os  # for the CARGO_TYPE headless override below

# Headless/driver override: CARGO_TYPE="3" answers the prompt automatically.
# When unset, the interactive input() below runs as before.
cargo_type = (
    os.environ.get("CARGO_TYPE", "").strip()
    or input(
        "Enter cargo type (1-4 or name): "
    ).strip()
)

print(
    "Selected cargo type:",
    cargo_type
)


In [ ]:
# ============================================================
# CELL 24 — DENSITY DATABASE
# ============================================================

density_database = {

    "scrap steel": {
        "density_kg_m3": 7850,
        "description": "Reference solid steel density"
    },

    "thermal coal": {
        "density_kg_m3": 800,
        "description": "Example bulk density"
    },

    "construction gravel": {
        "density_kg_m3": 1700,
        "description": "Example bulk density"
    },

    "agricultural grain": {
        "density_kg_m3": 750,
        "description": "Example bulk density"
    }
}

# Accept either the menu number (1-4) or the full cargo name.
_cargo_aliases = {
    "1": "scrap steel",
    "2": "thermal coal",
    "3": "construction gravel",
    "4": "agricultural grain",
}

cargo_key = _cargo_aliases.get(
    cargo_type.strip().lower(),
    cargo_type.strip().lower(),
)

if cargo_key in density_database:

    density = density_database[
        cargo_key
    ]["density_kg_m3"]

    print(
        "Selected density:",
        density,
        "kg/m³"
    )

else:

    density = None

    print(
        "Cargo type not found in density database.",
        f"(input: '{cargo_type}', lookup key: '{cargo_key}')"
    )


In [ ]:
# ============================================================
# CELL 25 — WEIGHT ESTIMATION
# ============================================================

# density comes from CELL 24 and is None when the cargo type is unknown.
density = globals().get("density")

_t25 = cargo_step("25. weight estimation")

if density is None:

    print(
        "Weight unavailable: no density for this cargo type."
    )
    cargo_done("25. weight estimation", _t25, status="unavailable", reason="no density")

elif volumes and volumes[0] is not None:

    relative_volume = volumes[0]

    estimated_weight = (
        relative_volume
        * density
    )

    print(
        "Estimated weight:",
        estimated_weight,
        "kg"
    )
    cargo_done("25. weight estimation", _t25, weight_kg=f"{estimated_weight:.1f}",
        density=density)

else:

    print(
        "Volume unavailable."
    )
    cargo_done("25. weight estimation", _t25, status="unavailable", reason="no volume")


In [ ]:
# ============================================================
# CELL 26 — FINAL INSPECTION RESULT
# ============================================================

result = {

    "inspection_status": "COMPLETED",

    "cargo_type": cargo_type,

    "volume": (
        float(volumes[0])
        if volumes[0] is not None
        else None
    ),

    "density_kg_m3": (
        float(density)
        if cargo_key in density_database
        else None
    ),

    "estimated_weight_kg": (
        float(estimated_weight)
        if "estimated_weight" in globals()
        else None
    ),

    "number_of_images": len(images),

    "segmentation": "SAM",

    "depth_estimation": "Depth Anything",

    "3d_processing": "Open3D"
}

print(
    json.dumps(
        result,
        indent=4
    )
)


In [ ]:
# ============================================================
# CELL 27 — SAVE INSPECTION RESULT
# ============================================================

result_path = (
    OUTPUT_DIR
    / "inspection_result.json"
)

with open(
    result_path,
    "w"
) as f:

    json.dump(
        result,
        f,
        indent=4
    )

print(
    "Result saved to:",
    result_path
)


In [ ]:
# ============================================================
# CELL 28 — INSPECTION RESULT LOCATION
# ============================================================

print("Inspection result saved to:")
print(result_path.resolve())

# ---- final, complete timing profile for the whole run -----------------------
# CELL 28 is the last cell, so this captures every cell that executed. The
# CELL 2 hook has already saved a snapshot after each of those cells; this is
# the authoritative final save, and the label the comparison below uses.
_timing_saved = cargo_save_timings(label="full run")

if _timing_saved:
    print("Timing report saved to:")
    for _key in ("json", "markdown", "history"):
        print(" -", _timing_saved[_key])

    # Diff against the previous full run when there is one.
    cargo_compare_timings(current=_timing_saved["payload"])
